In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import IPython

print("Версия Python:", sys.version)
print("Путь к интерпретатору:", sys.executable)
print('numpy :', np.__version__)
print('pandas:', pd.__version__)
print('seaborn:', sns.__version__)
print('IPython:', IPython.__version__)

In [ ]:
# Журнал решений: одна запись на одно решение
LOG_COLUMNS = [
    'field', 'issue', 'scope', 'n_rows',
    'value_before', 'action', 'value_after', 'rationale',
]

cleaning_log_entries = {}


def add_log(key, field, issue, scope, n_rows,
            value_before, action, value_after, rationale):
    """Добавляет решение; повторный вызов с тем же ключом обновляет запись"""
    cleaning_log_entries[key] = {
        'field': field,
        'issue': issue,
        'scope': scope,
        'n_rows': int(n_rows),
        'value_before': str(value_before),
        'action': action,
        'value_after': str(value_after),
        'rationale': rationale,
    }

Проверим расположение clients.csv и transactions.csv

In [ ]:
# Проверка наличия данных
from pathlib import Path

DATA_DIR = Path("data/raw")

CLIENTS_PATH = DATA_DIR / "clients.csv"
TRANSACTIONS_PATH = DATA_DIR / "transactions.csv"

print("Папка:", DATA_DIR.absolute())
print("clients.csv:", CLIENTS_PATH.is_file())
print("transactions.csv:", TRANSACTIONS_PATH.is_file())

## 1. Аудит качества данных: клиенты банка


## 1.1 Наивная загрузка
Читаем данные без разделителей и типов

In [ ]:
if CLIENTS_PATH.is_file() and TRANSACTIONS_PATH.is_file():
    df_clients_naive = pd.read_csv(CLIENTS_PATH, encoding='utf-8')
    df_transactions_naive = pd.read_csv(TRANSACTIONS_PATH, encoding='cp1251')
    for name, preview in [("clients", df_clients_naive),
                      ("transactions", df_transactions_naive)]:
        print(f"{name}: прочитано {len(preview)} строк, {preview.shape[1]} столбцов")
        display(preview.head(2))
    

Вывод: 
- clients.csv весь прочитан в таблицу (5702, 16)
- transactions.csv прочитан некорректно: получен один столбец вместо отдельных полей, видно, что разделителем является ';'

In [ ]:
# Типы всех столбцов
print("\n Типы данных всех столбцов:")
print(df_clients_naive.dtypes)

Вывод: типы требуют уточнения и не совсем корректны:
 - даты(birth_date, registration_date, card_open_date, app_last_login) представлены str, а не спец типом даты
  - satisfaction_score содержит целые оценки, а не дробные

## 1.2 Правильная загрузка clients
Зададим типы полей и даты разберём отдельно форматам источника


In [ ]:
from IPython.display import display
df_clients = pd.read_csv(
    CLIENTS_PATH,
    encoding='utf-8',
    dtype={
        'client_id': 'str',
        'city': 'str',
        'branch_code': 'str',  # Читаем как строку, так как в начале есть нули, которые при преобразовании в int станут незначащими и исчезнут
        'gender': 'category',
        'segment': 'category',
        'loyalty_tier': 'category',
        'card_type': 'category',
        'income_source': 'category',
        'has_mobile_app': 'category',
        'marketing_opt_in': 'category',
        'satisfaction_score': 'Int64',    # Это порядковая шкала, удобно Int так как порядок тогда уже задан 
        'monthly_income': 'float64',
    }
)

n_loaded = len(df_clients)

add_log(
    key='clients_read_types',
    field=('client_id; branch_code; gender; segment; loyalty_tier; card_type; income_source; has_mobile_app; marketing_opt_in; satisfaction_score; monthly_income'),
    issue='При загрузке необходимо задать смысловые типы столбцов',
    scope='Все строки clients.csv при корректном чтении',
    n_rows=n_loaded,
    value_before=('Представление в CSV; при пробном чтении branch_code интерпретировался как число'),
    action='Заданы типы при чтении',
    value_after=('Идентификаторы — строки; категориальные признаки — category; satisfaction_score — Int64; monthly_income — float64'),
    rationale=(
        'Сохранить ведущие нули кодов отделений, '
        'пропуски целочисленной оценки и смысловые типы признаков. '
        'Содержательные значения не подменяются.'
    ),
)

df_clients_raw = df_clients.copy(deep=True)

DATE_FORMATS = {
    "birth_date": "%Y-%m-%d",
    "registration_date": "%d.%m.%Y",
    "card_open_date": "%d.%m.%Y",
    "app_last_login": "%Y-%m",
}

rows = []
for col, fmt in DATE_FORMATS.items():
    raw = df_clients[col]
    parsed = pd.to_datetime(raw.str.strip(), format=fmt, errors="coerce")
    rows.append({
        "поле": col, "пропусков в CSV": raw.isna().sum(),
        "не распознано": (raw.notna() & parsed.isna()).sum(),
        "NaT после разбора": parsed.isna().sum(),
    })
    df_clients[col] = parsed

    

date_parse_audit = pd.DataFrame(rows).set_index("поле")
print(f"Размер: {df_clients.shape[0]} строк × {df_clients.shape[1]} столбцов")
display(date_parse_audit)
display(df_clients.head(3))

print("clients.shape:", df_clients.shape)

Сделаем одну таблицу с типом, количеством пропусков и их долей от всех значений в одной таблице

In [ ]:
overview = pd.DataFrame({
    "тип": df_clients.dtypes.astype(str),
    "уникальных": df_clients.nunique(),
    "пропусков": df_clients.isna().sum(),
    "пропуски, %": (df_clients.isna().mean() * 100).round(2),
})
display(overview)

## 1.3 Категории и отсутствующие сведения
## Аудит и нормализация категориальных данных

In [ ]:
# Список столбцов для анализа
cat_cols = [
    'gender', 'city', 'segment', 'loyalty_tier', 'card_type',
    'income_source', 'has_mobile_app', 'satisfaction_score', 'marketing_opt_in'
]

# Частота категорий в каждом столбце
for col in cat_cols:
    print(f"\n Столбец: {col}")
    print(df_clients[col].value_counts(dropna=False))  # dropna=False — показывает NaN тоже

Выводы: 
- Есть разные написания одних и тех же городов
- Нет мобильного приложения(has_mobile_app) у 2140 и у 2140 клиентов NaT в app_last_login, проверим совпадение этих групп
- 1549 клиентов не указали источник зарплаты. Нужно посмотреть опять же связи с другими признаками
- 648 клиента отказались от участия маркетинговой коммуникации(не ошибка), поэтому они не будут включены ни в одну из групп контактной кампании, но не исключаем их из общего анализа 
- 3416 пропуска в satisfaction_score нельзя ничем заменить. Далее проверим есть ли связи с другими признаками

## 1.3.1 Города: одно название
Уберём внешние и повторные пробелы, различия регистра, известные сокращения заменить по словарю.

In [ ]:
# Приведём все города к одному написанию
import re

def normalize_city(city):
    """Brings the city to its standard appearance"""
    if pd.isna(city):
        return city
    
    city = city.strip()  # убираем пробелы
    # Убираем "г. " или "город " в начале
    city = re.sub(r'^(г\.|город)\s*', '', city, flags=re.IGNORECASE)
    city = city.title() # Один регистр
    
    abbreviations = {
        'Спб': 'Санкт-Петербург',
        'Санкт Петербург': 'Санкт-Петербург',
        'Н.Новгород': 'Нижний Новгород',
        'Н. Новгород': 'Нижний Новгород',
        'Ростов-На-Дону': 'Ростов-на-Дону',
    }
    
    return abbreviations.get(city, city)

city_before = df_clients['city'].copy()
city_after = city_before.apply(normalize_city)

# Учитываем только действительно изменившиеся значения
city_changed = (city_before.astype('string').fillna('')
        .ne(city_after.astype('string').fillna(''))
)

df_clients['city'] = city_after

add_log(
    key='clients_city_normalization',
    field='city',
    issue='Несколько написаний одного города',
    scope='normalize_city(city) отличается от исходного значения',
    n_rows=city_changed.sum(),
    value_before=( f'{city_before.nunique()} написаний; например «г. Москва», «МОСКВА», «СПб», «Санкт Петербург»'),
    action='Нормализация',
    value_after=f'{city_after.nunique()} единых названий городов',
    rationale=('Унифицированы регистр, префиксы и сокращения. Географический смысл сохранён; n_rows — число изменённых значений до удаления дублей.'),
)

print(df_clients['city'].value_counts(dropna=False))

Вывод: разные написания одного и того же города привели к одному названию, в итоги получили 6 городов

## 1.3.2 Приложение и месяц входа
Равенство числа клиентов без приложения и числа отсутствующих дат ещё не доказывает, что это одни и те же строки. Проверим сочетания признаков напрямую.

In [ ]:
# Проверяет соответствие отсутствия даты последнего входа и отсутствие мобильного приложения 
mask_anomaly = (
    (df_clients['has_mobile_app'] == 'Нет') & (df_clients['app_last_login'].notna())  
    |
    (df_clients['has_mobile_app'] == 'Да') & (df_clients['app_last_login'].isna())    
)

df_anomalies = df_clients[mask_anomaly].copy()

print(f"Найдено аномальных записей: {len(df_anomalies)}")
print(f"Из {len(df_clients)} всего ({len(df_anomalies)/len(df_clients)*100:.2f}%)")

# Смотрим на аномалии
display(df_anomalies[['client_id', 'has_mobile_app', 'app_last_login']].head(20))

# Тепловая карта
app_login_check = pd.crosstab(
    df_clients['has_mobile_app'],
    df_clients['app_last_login'].isna().map({True: 'Нет даты', False: 'Есть дата'})
).reindex(columns=['Есть дата', 'Нет даты'], fill_value=0)

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.heatmap(app_login_check, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
ax.set(title='Приложение и дата входа', xlabel='Дата последнего входа', ylabel='Есть приложение')
ax.grid(False)
plt.tight_layout()
plt.show()

Вывод: даты последнего входа не указаны только у людей, у которых нет приложения. Данный пропуск ничем не заменяем

## 1.4 Повторные записи
В клиентском справочнике единица наблюдения — клиент, а client_id должен быть уникален. Разделим две ситуации:

 - полностью совпадающие записи — лишние копии одной анкеты;

 - одинаковый client_id, но разные значения полей — противоречивые версии анкеты.

## 1.4.1 Удаление полных копий

In [ ]:
n_clients_raw = len(df_clients)
n_unique_clients_raw = df_clients["client_id"].nunique()
# Проверяем, что нет пустых client_id — иначе нельзя корректно считать дубликаты
missing_id = (df_clients["client_id"].isna() | df_clients["client_id"].str.strip().eq("").fillna(False))
if missing_id.any():
    raise ValueError("Найдены пустые client_id: сначала разберите эти строки отдельно.")

fully_duplicated_rows = df_clients.duplicated().sum()
display(pd.Series({
    "строк до удаления повторов": n_clients_raw,
    "уникальных client_id": n_unique_clients_raw,
    "строк с повторяющимся ID": df_clients["client_id"].duplicated(keep=False).sum(),
    "лишних полных копий": fully_duplicated_rows,
}, name="количество").to_frame())

# Сохраняем удаляемые копии в отдельный df для журнала и последующей проверки
clients_full_duplicates = df_clients.loc[df_clients.duplicated()].copy()
df_clients = df_clients.drop_duplicates().copy()
n_after_full = len(df_clients)
n_full_removed = n_clients_raw - n_after_full

add_log(
    key='clients_full_duplicates', field='clients.*',
    issue='Полностью совпадающие клиентские анкеты',
    scope='df_clients.duplicated(keep="first") после нормализации',
    n_rows=n_full_removed, value_before=f'{n_clients_raw} строк',
    action='Удалены полные дубликаты', value_after=f'{n_after_full} строк',
    rationale=('Единица наблюдения — клиент. Полные копии анкеты не должны увеличивать его вес. Сохранено первое вхождение каждой одинаковой строки.'),
)

print(f"Удалено полных копий: {n_full_removed}; осталось строк: {n_after_full}")

Вывод: обнаружено 40 полный дубликатов, которые были удалены(слишком маловероятно открытие двух абсолютно одинаковых карт в один день одним пользователем).
 
Остальные частичные дубликаты с одинаковым client_id нужно проверить: какие признаки отличаются(т.е. возможно ли что это 2 или более карт у одного клиента)

## 1.4.2 Поиск конфликтующих строк

In [ ]:
# Проверяем все неидентичные записи клиентов с повторяющимся ID
conflict_mask = df_clients['client_id'].duplicated(keep=False)

clients_conflicts = (
    df_clients.loc[conflict_mask]
    .sort_values('client_id')
    .copy()
)

print('Конфликтующих строк:', len(clients_conflicts))
print('Клиентов с конфликтом:',
      clients_conflicts['client_id'].nunique())

# Количество разных значений каждого поля внутри client_id
different = clients_conflicts.groupby('client_id', observed=True).nunique(dropna=False).gt(1)

# Число клиентов с расхождениями по каждому полю
conflict_counts = different.sum()

display(
    conflict_counts[conflict_counts > 0]
    .to_frame('n_clients_with_conflict')
)

conflict_cols = different.columns[different.any()].tolist()
display(clients_conflicts[['client_id'] + conflict_cols])

Вывод: После удаления полных копий остались 30 строк, относящихся к 15 client_id, отличия: 13 по городам и 15 по сегментам, так как остальные поля совпадают.
Удалять это строки нет оснований, так это клиент мог поменять со временем свои данные, но и определить какая запись сейчас действительна невозможно.
Решение: Сохраняем по одной строке на клиента. У всех 15 клиентов city и segment заменяем пропусками, так как нет вариантов определить какая запись более актуальная

## 1.4.3 Объединение строк(решение конфликтов)

In [ ]:
# Берём исходные конфликтующие анкеты
clients_conflicts_original = clients_conflicts.copy()
resolved_clients = clients_conflicts_original.copy()

# Определяем, какие поля расходятся у каждого клиента
different = clients_conflicts_original.groupby('client_id', observed=True).nunique(dropna=False).gt(1)


conflict_cols = different.columns[different.any()].tolist()
conflict_ids = clients_conflicts_original['client_id'].unique()

# Заменяем только противоречивые поля в исходных анкетах
for col in conflict_cols:
    ids = different.index[different[col]]

    selected = resolved_clients['client_id'].isin(ids)

    # Сохраняем примеры до замены
    before_examples = (
        resolved_clients.loc[selected, col]
        .astype('string')
        .drop_duplicates()
        .tolist()
    )

    resolved_clients.loc[selected, col] = pd.NA

    add_log(
        key=f'clients_conflict_{col}', field=col,
        issue='Противоречивые значения у одного client_id',
        scope='Клиенты с несколькими значениями поля среди конфликтующих анкет',
        n_rows=selected.sum(), value_before=before_examples,
        action='Заменено пропуском',
        value_after=f'NA в {int(selected.sum())} исходных версиях анкет; {len(ids)} уникальных клиентов',
        rationale='Достоверную версию определить нельзя. Заменяется только противоречивое поле; согласованные признаки клиента сохранены. n_rows — число затронутых версий до объединения.',
    )

# После устранения противоречий объединяем версии анкет в одну запись на клиента
n_versions_before = len(resolved_clients)

resolved_clients = resolved_clients.drop_duplicates()

n_versions_removed = n_versions_before - len(resolved_clients)

add_log(
    key='clients_conflict_versions_merge', field='client_id; clients.*',
    issue='После устранения противоречий версии анкеты совпадают',
    scope='resolved_clients после замены конфликтующих полей',
    n_rows=n_versions_removed,
    value_before=f'{n_versions_before} версий; {resolved_clients["client_id"].nunique()} клиентов',
    action='Объединены версии; удалены лишние копии',
    value_after=f'{len(resolved_clients)} анкет',
    rationale='После устранения противоречий сохраняем одну анкету на клиента, не удаляя самого клиента из базы.',
)



# Заменяем прежний результат обработки исправленными анкетами
df_clients = pd.concat(
    [
        df_clients.loc[~df_clients['client_id'].isin(conflict_ids)],
        resolved_clients,
    ],
    ignore_index=True,
)

print('Анкеты после обработки:')
display(
    df_clients.loc[
        df_clients['client_id'].isin(conflict_ids),
        ['client_id', 'city', 'segment'],
    ].sort_values('client_id')
)
print( "Осталось уникальных клиентов:", len(df_clients))

Вывод: После объединения 30 конфликтных анкет в 15 записей справочник содержит 5647 уникальных client_id. Удалены лишние представления клиента, а не сами клиенты.

## 1.5 Возраст и согласованность дат
Справочник представлен на момент 30.06.2025

In [ ]:
REFERENCE_DATE = pd.Timestamp('2025-06-30')

date_cols = [
    'birth_date',
    'registration_date',
    'card_open_date',
    'app_last_login',
]
for col in date_cols:
    assert pd.api.types.is_datetime64_any_dtype(df_clients[col]), f'{col}: нужен datetime'


def full_years(birth_date, event_date):
    """
    Calculates the age in full years between the date of birth and the event.
    При неизвестной дате возвращает пропуск.
    """
    # Проверяем, наступил ли день рождения в году события
    before_birthday = (
        (event_date.dt.month < birth_date.dt.month)
        | ((event_date.dt.month == birth_date.dt.month)
           & (event_date.dt.day < birth_date.dt.day))
    )

    years = event_date.dt.year - birth_date.dt.year - before_birthday.astype(int)

    return years.where(birth_date.notna() & event_date.notna()).astype('Int64')


# Создаём Series с контрольной датой для каждого клиента
reference_dates = pd.Series(REFERENCE_DATE, index=df_clients.index)

for target, event in {
    'age': reference_dates,
    'age_at_registration': df_clients['registration_date'],
    'age_at_card_open': df_clients['card_open_date'],
}.items():
    df_clients[target] = full_years(df_clients['birth_date'], event)
    
add_log(
    key='clients_age_features', field='age; age_at_registration; age_at_card_open',
    issue='Для анализа нужны возрасты на дату среза и событий',
    scope='Все клиенты после обработки повторных анкет',
    n_rows=len(df_clients), value_before='Производные возрастные признаки отсутствовали',
    action='Созданы признаки',
    value_after=f'Полные годы на {REFERENCE_DATE:%d.%m.%Y}, на дату регистрации и открытия карты',
    rationale='Возраст рассчитан с учётом наступления дня рождения. Исходные даты не изменяются.',
)


age_fields = ['age', 'age_at_registration', 'age_at_card_open']
display(df_clients[age_fields].describe().round(1))
display(df_clients[date_cols].isna().sum().to_frame('неизвестна дата'))

age = df_clients['age'].dropna()

# Диаграмма распределения клиентов по возрасту
plt.figure(figsize=(12, 5))
# discrete=True рисует отдельные столбцы для каждого целого возраста, а не сплошную кривую
sns.histplot(age, discrete=True, color='skyblue', edgecolor='black')
plt.title('Распределение клиентов по возрасту (полных лет)')
plt.xlabel('Возраст')
plt.ylabel('Количество клиентов')
plt.grid(axis='y', alpha=0.7)
plt.tight_layout()
plt.show()

# Столбцы для ручного просмотра анкет
age_cols = [
    'client_id', 'birth_date', 'age',
    'registration_date', 'age_at_registration',
    'card_open_date', 'age_at_card_open', 'card_type',
]

# Самые молодые
display(df_clients.sort_values('age')[age_cols].head(5))
# Самые пожилые
display(df_clients.sort_values('age', ascending=False)[age_cols].head(5))

Вывод: возраст клиентов на 30.06.2025 составляет от 23 до 88 лет. Возраста на данный момент полностью подходят для любых операций. Но нужно ещё проверить их возраст на момент регистрации и открытия карты.
Возраст 88 на диаграмме выглядит анамально и не всписывается в общую форму. Его нужно исследовать подробнее, пока нельзя сказать, что это точно ошибка

## 1.5.1 Проверка аномального возраста 88

In [ ]:
# диагностический флаг
AGE_TO_CHECK = 88

df_clients['age_88_flag'] = (
    df_clients['age'].eq(AGE_TO_CHECK).fillna(False)
)

add_log(
    key='clients_age88_flag',
    field='age_88_flag',
    issue='Нужно выделить скопление наблюдений возраста 88 лет',
    scope='Все клиенты; True при age == AGE_TO_CHECK',
    n_rows=len(df_clients),
    value_before='Флаг отсутствовал',
    action='Создан диагностический флаг',
    value_after=f'True у {int(df_clients["age_88_flag"].sum())} клиентов',
    rationale='Возраст сам по себе возможен. Флаг позволяет проверить группу отдельно без удаления клиентов и изменения даты рождения.',
)

add_log(
    key='clients_age88_retained',
    field='birth_date; age',
    issue='Скопление наблюдений на максимальном возрасте',
    scope=f'age == {AGE_TO_CHECK}',
    n_rows=df_clients['age_88_flag'].sum(),
    value_before=f'Возраст {AGE_TO_CHECK} лет',
    action='Оставлено',
    value_after='Даты рождения и возраст сохранены',
    rationale='Подтверждения ошибки нет. Повышенная частота значения не является достаточным основанием для удаления.',
)


age_88 = df_clients.loc[df_clients['age_88_flag']].copy()

print(f'Возраст {AGE_TO_CHECK} лет: {len(age_88)} клиентов ({df_clients["age_88_flag"].mean():.2%} всей базы)')

# Проверяем, не связан ли всплеск с повторением одной даты рождения
print('Разных дат рождения в группе:', age_88['birth_date'].nunique())

display(
    age_88['birth_date']
    .value_counts()
    .head(10)
    .to_frame('клиентов')
)

# Смотрим даты и возраст на момент прошлых событий
display(
    age_88[ ['client_id', 'birth_date', 'age', 'registration_date', 'age_at_registration', 'card_open_date', 'age_at_card_open']]
    .sort_values('birth_date')
    .head(15)
)

Вывод: Даты рождения, регистрации и открытия карты все очень разные, что не похоже на заглушку. Но установить причину не удалось. Люди 88 возраста составляют 2, 18 % от всех.
Решение:
 - сохраняем клиентов
 - оставляем диагностический флаг(age_88_flag)

## 1.5.2 Проверим возрастные рамки и сочетание возраста на момент различных действий

In [ ]:
birth = df_clients['birth_date']
reg = df_clients['registration_date']
card = df_clients['card_open_date']
login_month = df_clients['app_last_login'].dt.to_period('M')

age_flags = pd.DataFrame({
    'birth_in_future': birth.gt(REFERENCE_DATE),
    'age_over_110': df_clients['age'].gt(110),
    'age_under_14': df_clients['age'].lt(14),
    'card_open_under_14': df_clients['age_at_card_open'].lt(14),
    'credit_card_under_18': (df_clients['card_type'].eq('Кредитная')
                           & df_clients['age_at_card_open'].lt(18)),
}).fillna(False)
date_flags = pd.DataFrame({
    'registration_before_birth': reg.lt(birth),
    'card_before_birth': card.lt(birth),
    'registration_in_future': reg.gt(REFERENCE_DATE),
    'card_open_in_future': card.gt(REFERENCE_DATE),
    'card_before_registration': card.lt(reg),
    'login_after_snapshot_month': login_month.gt(REFERENCE_DATE.to_period('M')),
    'login_before_registration_month': login_month.lt(reg.dt.to_period('M')),
    'login_before_birth_month': login_month.lt(birth.dt.to_period('M')),
}).fillna(False)
checks = pd.concat([age_flags, date_flags], axis=1)
display(checks.sum().to_frame('клиентов с сигналом'))
problem_clients = df_clients.loc[checks.any(axis=1)].copy()
view_cols = ['client_id', *date_cols, *age_fields, 'card_type']


add_log(
    key='clients_credit_under_18_retained',
    field='card_type; card_open_date; age_at_card_open',
    issue='Кредитная карта открыта в возрасте до 18 лет',
    scope='age_flags["credit_card_under_18"] == True',
    n_rows=age_flags['credit_card_under_18'].sum(),
    value_before='Тип текущей карты — кредитная; возраст открытия ниже 18',
    action='Оставлено',
    value_after='Тип карты и даты сохранены; случаи доступны в аудите',
    rationale='Не установлено, какое поле ошибочно или отличается по смыслу. Не исправлять значения предположением; учитывать ограничение при анализе возраста открытия.',
)

add_log(
    key='clients_early_login_retained',
    field='app_last_login; registration_date',
    issue='Месяц последнего входа раньше месяца регистрации',
    scope='date_flags["login_before_registration_month"] == True',
    n_rows=date_flags['login_before_registration_month'].sum(),
    value_before='Месяц входа < месяца регистрации',
    action='Оставлено',
    value_after='Даты сохранены; противоречия доступны в аудите',
    rationale='Достоверное значение неизвестно. Не подменять даты; сравнение учитывает месячную точность app_last_login.',
)


# сохраняем флаги отдельно
client_date_audit = checks.set_axis(df_clients['client_id'], axis=0)


Вывод: 
 - у 3 клиентов на данный момент тип карты указан как "кредитная", а возраст при открытии этой карты составляет 16-17 лет. Это может свидетельствовать либо о несогласованности атрибутов, либо о правонарушении, либо об эмансипации.
 - у 15 клиентов месяц последнего входа предшествует месяцу регистрации. 
Проанализируем подробнее:

In [ ]:
# Два диагностических графика на одном рисунке
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Возраст при открытии кредитной карты
credit = df_clients.loc[
    df_clients['card_type'].eq('Кредитная')
    & df_clients['age'].notna()
    & df_clients['age_at_card_open'].notna()
].copy()

young = credit['age_at_card_open'].lt(18)

for mask, color, label, size in [
    (~young, 'blue', '18 лет и старше', 15),
    (young, 'crimson', 'Младше 18 лет', 65),
]:
    axes[0].scatter(
        credit.loc[mask, 'age'].astype(float),
        credit.loc[mask, 'age_at_card_open'].astype(float),
        color=color, s=size, alpha=0.6,
        label=f'{label} (n={int(mask.sum())})',
    )

axes[0].axhline(18, color='darkorange', linestyle='--', label='Ориентир: 18 лет')

# Подпишем только несколько подозрительных точек
for _, row in credit.loc[young].head(5).iterrows():
    axes[0].annotate(
        row['client_id'],
        (float(row['age']), float(row['age_at_card_open'])),
        xytext=(5, 5), textcoords='offset points', fontsize=8,
    )

axes[0].set(
    title='Возраст при открытии кредитной карты',
    xlabel='Возраст на дату среза, лет',
    ylabel='Возраст при открытии, лет',
)
axes[0].legend(fontsize=8)

# 2. Месяц регистрации и месяц последнего входа
timeline = df_clients.loc[
    df_clients['registration_date'].notna()
    & df_clients['app_last_login'].notna()
].copy()

registration_month = timeline['registration_date'].dt.to_period('M')
last_login_month = timeline['app_last_login'].dt.to_period('M')
early_login = last_login_month.lt(registration_month)

# На графике обе даты представлены началом соответствующего месяца
x = registration_month.dt.to_timestamp()
y = last_login_month.dt.to_timestamp()

for mask, color, label, size in [
    (~early_login, 'blue', 'Вход не раньше регистрации', 15),
    (early_login, 'crimson', 'Вход раньше регистрации', 65),
]:
    axes[1].scatter(
        x.loc[mask], y.loc[mask],
        color=color, s=size, alpha=0.6,
        label=f'{label} (n={int(mask.sum())})',
    )

if not timeline.empty:
    lower = min(x.min(), y.min())
    upper = max(x.max(), y.max())

    axes[1].plot(
        [lower, upper], [lower, upper],
        color='darkorange', linestyle='--',
        label='Один месяц',
    )

axes[1].set(
    title='Регистрация и последний вход в приложение',
    xlabel='Месяц регистрации',
    ylabel='Месяц последнего входа',
)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Временно показываем все столбцы анкет
with pd.option_context('display.max_columns', None):
    print('Вход раньше месяца регистрации:')
    display(df_clients.loc[date_flags['login_before_registration_month']].sort_values('client_id').head(15))
    print('Кредитная карта открыта до 18 лет:')
    display(df_clients.loc[age_flags['credit_card_under_18']].sort_values('client_id').head(3))

Вывод: графики показывают красными точкам клиентов, которые имееют несоответствие. Решение: 
- клиентов сохраняем
- даты не исправляем
- сохраняем флаги
- 15 клиентов со спорным месяцем входа не будет использовать для выводов об активности

## 1.6 Проверка ежемесячного дохода

Цель - отделить некорректные значения от редких, но возможных доходов.
Сначала проверим значения и форму распределения, затем решим, какие наблюдения изменять, а какие оставить с диагностическим флагом.


In [ ]:
LOW_INCOME_LIMIT = 10_000

# Сохраняем значения до замены для сравнения
income_before = df_clients['monthly_income'].copy()
n_minus999 = income_before.eq(-999).sum()

# Другие отрицательные значения требуют отдельного решения
other_negative = income_before.lt(0) & income_before.ne(-999)
assert not other_negative.any(), 'Найдены отрицательные доходы, отличные от −999. Перед продолжением нужно проверить их отдельно.'

# Заменяем некорректный код пропуском, не удаляя клиентов
df_clients['monthly_income'] = income_before.replace(-999, np.nan)
income = df_clients['monthly_income']

add_log(
    key='clients_income_minus999',
    field='monthly_income',
    issue='Отрицательное значение −999 в поле дохода',
    scope='monthly_income == -999 после обработки повторов',
    n_rows=n_minus999,
    value_before=f'−999; пропусков до обработки: {int(income_before.isna().sum())}',
    action='Заменено пропуском',
    value_after=f'NaN; всего пропусков: {int(income.isna().sum())}',
rationale=('Крупная сумма сама по себе не доказывает ошибку. На этом этапе верхнее ограничение не применяется. Значения сохранены для последующего анализа контекста клиентов и чувствительности описательных показателей.'),
)

df_clients['income_missing'] = income.isna()
df_clients['income_low_flag'] = (income.gt(0) & income.lt(LOW_INCOME_LIMIT)).fillna(False)

add_log(
    key='clients_income_missing_flag',
    field='income_missing',
    issue='Нужно отмечать отсутствие суммы после очистки',
    scope='Все клиенты; monthly_income.isna()',
    n_rows=len(df_clients),
    value_before='Флаг отсутствовал',
    action='Создан флаг',
    value_after=f'True у {int(df_clients["income_missing"].sum())} клиентов',
    rationale='Отделить отсутствие суммы от её размера. Учитываются исходные пропуски и заменённые −999.',
)

add_log(
    key='clients_income_low_flag',
    field='income_low_flag',
    issue='Нужно выделить низкие положительные доходы',
    scope=f'Все клиенты; 0 < monthly_income < {LOW_INCOME_LIMIT}',
    n_rows=len(df_clients),
    value_before='Флаг отсутствовал',
    action='Создан диагностический флаг',
    value_after=f'True у {int(df_clients["income_low_flag"].sum())} клиентов',
    rationale='Порог используется для дополнительной проверки и анализа чувствительности, а не как доказательство ошибки. Пропуски не относятся к низким доходам.',
)


# Сводная таблица до и после обработки
income_check = pd.DataFrame({
    'До обработки': [income_before.notna().sum(), income_before.isna().sum(), n_minus999],
    'После обработки': [income.notna().sum(), income.isna().sum(), income.eq(-999).sum()],
}, index=['Доход заполнен', 'Доход отсутствует', 'Значение −999'])
display(income_check)

add_log(
    key='clients_low_income_retained',
    field='monthly_income',
    issue='Необычно низкие положительные суммы',
    scope=f'0 < monthly_income < {LOW_INCOME_LIMIT}',
    n_rows=df_clients['income_low_flag'].sum(),
    value_before=f'Положительные суммы ниже {LOW_INCOME_LIMIT} ₽',
    action='Оставлено',
    value_after='Суммы сохранены; группа отмечена income_low_flag',
    rationale='Причина низких сумм не установлена. Не умножать на 1000 и не заменять медианой. Влияние проверяется сравнением результатов с группой и без неё.',
)

add_log(
    key='clients_high_income_retained',
    field='monthly_income',
    issue='Доходы выше 1 млн ₽',
    scope='monthly_income > 1000000',
    n_rows=income.gt(1_000_000).sum(),
    value_before='Доходы выше 1000000 ₽',
    action='Оставлено',
    value_after='Суммы сохранены без ограничения сверху',
    rationale='Проверен контекст клиентов; подтверждения ошибки нет. Крупная сумма сама по себе не основание для удаления.',
)

print(f"Пропусков после обработки: {df_clients['income_missing'].sum()} ({df_clients['income_missing'].mean():.2%} базы)")
print(f"Положительных доходов ниже {LOW_INCOME_LIMIT:,} ₽: {df_clients['income_low_flag'].sum()}")

# Для анализа распределения берём доходы без пропусков
income_known = df_clients['monthly_income'].dropna().copy()

p95 = income_known.quantile(0.95)
income_for_plot = income_known.loc[income_known.le(p95)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_samples = [
    (axes[0], income_known, 'Все известные доходы'),
    (axes[1], income_for_plot, f'Увеличенный участок: доходы до P95 ({p95:,.0f} ₽)'),
]

# Строим два графика: полный диапазон и обрезанный по 95-му перцентилю
for ax, values, title in plot_samples:
    sns.histplot(x=values, bins=50, edgecolor='white', ax=ax)
    # На обеих панелях — характеристики полной выборки известных доходов
    ax.axvline(income_known.median(), color='crimson', linestyle='--', label=f'Медиана всех доходов: {income_known.median():,.0f} ₽')
    ax.axvline(income_known.mean(), color='darkorange', linestyle='-.', label=f'Среднее всех доходов: {income_known.mean():,.0f} ₽')
    ax.axvline(LOW_INCOME_LIMIT, color='gray', linestyle=':', label='Порог проверки: 10 000 ₽')
    ax.set(title=f'{title}\nn = {len(values)}', xlabel='Ежемесячный доход, ₽', ylabel='Количество клиентов')
    ax.ticklabel_format(axis='x', style='plain')
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

print(f'Только на правой панели не показаны доходы выше P95: {income_known.gt(p95).sum()}. Из df_clients эти наблюдения не удалены.')

Решение: 
- Значения −999 заменены пропусками, так как очень похожи на значения(флаг) которые кто-то ставил при отсутствии дохода. Достоверно определить нельзя.
- 175 положительных доходов ниже 10'000 отмечены флагом(income_low_flag) (Похоже единицы измерения когда-то перепутали и забыли написать, что это тыщи, так как по форме распределения это значения не вписываются

## 1.6.1. Описательная статистика

Рассчитаем квантили и характеристики распределения
по всем известным доходам.

Покажем результаты без значений ниже 10 000 ₽.

In [ ]:
# Сравниваем расширенную описательную статистику полной выборки и выборки без низких доходов
income_without_low = income.loc[
    income.notna() & ~df_clients['income_low_flag']
]

samples = {
    'Все известные доходы': income_known,
    'Без доходов ниже 10 000 ₽': income_without_low,
}

rows = {}

for name, values in samples.items():
    row = values.describe( percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    row['IQR'] = values.quantile(0.75) - values.quantile(0.25)
    row['Асимметрия'] = values.skew()
    row['Избыточный эксцесс'] = values.kurt()
    rows[name] = row

income_stats = pd.DataFrame(rows).rename(index={
    'count': 'Количество',
    'mean': 'Среднее',
    'std': 'Стандартное отклонение',
    'min': 'Минимум',
    '1%': 'P1',
    '5%': 'P5',
    '25%': 'Q1',
    '50%': 'Медиана',
    '75%': 'Q3',
    '95%': 'P95',
    '99%': 'P99',
    'max': 'Максимум',
})

display(income_stats.round(2))

# Оцениваем изменение основных показателей
sensitivity = pd.DataFrame({
    'Все известные доходы': [income_known.mean(), income_known.median()],
    'Без низких доходов': [income_without_low.mean(), income_without_low.median()],
}, index=['Среднее', 'Медиана'])

sensitivity['Изменение, %'] = (sensitivity['Без низких доходов']
                               .div(sensitivity['Все известные доходы'])
                               .sub(1)
                               .mul(100))
display(sensitivity.round(2))

Вывод: длинный правых хвост, поэтому прежде всего будем использовать медиану и квартили
Решение: клиентов с низкими доходами не удаляем. Используем выборку без низких доходов для дальнейшего анализа.

## 1.6.2. Проверка крайних значений

Посмотрим на минимальные положительные и максимальные доходы
вместе с источником дохода и сегментом клиента.

In [ ]:
# Считаем количество наблюдений на границах распределения (нули, экстремумы)
income_cols = ['client_id', 'monthly_income', 'income_source', 'segment', 'loyalty_tier', 'city']

income_limits = pd.Series({
    'Отрицательные значения после обработки': income.lt(0).sum(),
    'Нулевые доходы': income.eq(0).sum(),
    'Положительные доходы ниже 10 000 ₽': df_clients['income_low_flag'].sum(),
    'Доходы выше 1 000 000 ₽': income.gt(1_000_000).sum(),
}, name='Количество')

display(income_limits.to_frame())

# Выводим примеры анкет с минимальными и максимальными доходами для ручной проверки
print('Пять минимальных положительных доходов:')
display(df_clients.loc[income.gt(0)].nsmallest(5, 'monthly_income')[income_cols])

print('Пять максимальных доходов:')
display(df_clients.nlargest(5, 'monthly_income')[income_cols])

Решение: 
- низкие положительные доходы сохраняем с income_low_flag;
- крупные доходы не ограничиваем сверху;
- ничем не заменяем ни медианой, ни умножением на 1000, так как ошибку не получается подтвердить.

## 1.6.3. В каких группах встречаются низкие доходы

Добавлено: проверим распределение income_low_flag
по источникам дохода и клиентским сегментам.

In [ ]:
# профиль низких доходов по группам
low_income_profiles = {}

for group in ['income_source', 'segment']:
    profile = (
        df_clients.groupby(group, observed=True, dropna=False)
        .agg(
            n_clients=('client_id', 'size'),
            income_known=('monthly_income', 'count'),
            low_income=('income_low_flag', 'sum'),
        )
    )

    # Рассчитываем долю низких доходов среди известных, защищаясь от деления на ноль
    profile['low_among_known_pct'] = (
        profile['low_income']
        .div(profile['income_known'].replace(0, np.nan))
        .mul(100)
    )

    low_income_profiles[group] = profile

    print(f'\nГруппировка: {group}')
    display(profile.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

titles = {
    'income_source': 'По источнику дохода',
    'segment': 'По сегменту',
}

for ax, (group, profile) in zip(axes, low_income_profiles.items()):
    plot_data = (
        profile.loc[profile['income_known'].gt(0)]
        .sort_values('low_among_known_pct')
    )

    plot_data = profile.loc[profile['income_known'].gt(0)].sort_values('low_among_known_pct')
    labels = [f'{name if pd.notna(name) else "Пропуск"} (n={int(row["income_known"])})' for name, row in plot_data.iterrows()]
    bars = ax.barh(labels, plot_data['low_among_known_pct'])
    

    # Подписываем значения на столбцах и ограничиваем ось X для лучшей читаемости
    ax.bar_label(bars, labels=[f'{value:.1f}%' for value in plot_data['low_among_known_pct']], padding=4)
    ax.set(title=titles[group], xlabel='Доля низких доходов среди известных, %', ylabel='')
    ax.set_xlim(0, max(5, plot_data['low_among_known_pct'].max() * 1.3))
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

Вывод: закономерностей не найдено

## 1.6.4. Форма распределения и Q–Q-графики

Сравним положительные доходы с нормальным распределением:
в исходной шкале и после логарифмирования.

In [ ]:
from scipy import stats

positive = income.loc[income.gt(0)].astype(float)

positive_without_low = positive.loc[~df_clients.loc[positive.index,'income_low_flag']]

shape_samples = {
    'Доход, все положительные': positive,
    'ln(дохода), все положительные': np.log(positive),
    'ln(дохода), без низких': np.log(positive_without_low),
}

shape_comparison = pd.DataFrame({
    name: {
        'Количество': len(values),
        'Асимметрия': values.skew(),
        'Избыточный эксцесс': values.kurt(),
    }
    for name, values in shape_samples.items()
}).T

display(shape_comparison.round(3))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# Строим Q-Q графики через scipy.stats.probplot, выделяя цветом клиентов с низким доходом относительно опорной прямой
for ax, (name, values) in zip(axes, shape_samples.items()):
    values = values.dropna().sort_values()

    if len(values) < 4 or values.nunique() < 2:
        ax.set_title(name)
        ax.text(
            0.5, 0.5, 'Недостаточно данных',
            ha='center', va='center',
            transform=ax.transAxes,
        )
        continue

    # получаем квантили и параметры опорной прямой
    (theoretical, observed), (slope, intercept, _) = (
        stats.probplot(values.to_numpy(), dist='norm')
    )

    # выделяем низкие доходы
    low_flag = df_clients.loc[values.index, 'income_low_flag'].to_numpy(dtype=bool)

    for mask, color, label in [
        (~low_flag, 'steelblue', 'Доход от 10 000 ₽'), 
        (low_flag, 'crimson', 'Доход ниже 10 000 ₽')
    ]:
        if mask.any():
            ax.scatter(theoretical[mask], observed[mask], s=10, alpha=0.5, color=color, label=label)

    ax.plot(theoretical, slope * theoretical + intercept, color='darkorange', label='Опорная прямая')
    ax.set(title=f'{name}\nn = {len(values)}', xlabel='Теоретические квантили N(0, 1)', ylabel='Выборочные квантили')
    ax.legend(fontsize=7)
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

Вывод: изначально есть правосторонняя асимметрия. После логарифмирования асимметрия становится отрицательной из-за низких доходов, и после их отрезания видно более близкое положение точек к прямой(норм распред)

## Итоги
  - Значения -999 заменены на пропуски. 
  - Созданы столбцы с флагами 
    - income_missing — отсутствие дохода после обработки (10,87% клиентов)
    - income_low_flag — положительный доход ниже 10 000 (175 клиентов)
  - логарифмирование приближает распределения дохода к нормальному, без учёта низких доходов
Дальше:
 1) Клиенты м низким доходом не удалены, используем их для анализа, где доход не важен
2) Результаты, зависящие отдохода, лучше сравнивать с вариантом с и без income_low_flag

## 1.7 Исследование отсутствующих сведений
Для более удобного дальнейшего анализа добавим столбцы: 
 - income_source_missing(источник дохода не указан)
 - score_missing (отсутствует оценка)
 - income_missing (ежемесячных доход не указан)
Цель — оценить можно ли что-то заполнить и выбрать правила как дальше анализировать

In [ ]:
# Создаем флаг отсутствия "не указан"
source_text = (
    df_clients['income_source']
    .astype('string')
    .str.strip()
    .str.casefold()
)

df_clients['income_source_missing'] = (
    source_text.isna()
    | source_text.isin(['', 'не указан'])
)

df_clients['score_missing'] = df_clients['satisfaction_score'].isna()
df_clients['city_missing'] = df_clients['city'].isna()
df_clients['segment_missing'] = df_clients['segment'].isna()

# Три основных показателя для дальнейших сравнений
flags = [
    'income_missing',
    'income_source_missing',
    'score_missing',
]

flag_labels = {
    'income_missing': 'Сумма дохода',
    'income_source_missing': 'Источник дохода',
    'score_missing': 'Оценка',
    'city_missing': 'Город',
    'segment_missing': 'Сегмент',
}

summary_flags = flags + ['city_missing', 'segment_missing']

missing_summary = pd.DataFrame({
    'Отсутствует': df_clients[summary_flags].sum(),
    'Доля базы, %': df_clients[summary_flags].mean().mul(100),
}).rename(index=flag_labels)

display(missing_summary.round(2))

new_missing_flags = {
    'income_source_missing': ('income_source', 'NaN, пустая строка или «Не указан» после удаления краевых пробелов и приведения к нижнему регистру'),
    'score_missing': ('satisfaction_score', 'satisfaction_score.isna()'),
    'city_missing': ('city', 'city.isna()'),
    'segment_missing': ('segment', 'segment.isna()'),
}

for flag, (source_field, rule) in new_missing_flags.items():
    add_log(
        key=f'clients_create_{flag}',
        field=flag,
        issue=f'Нужно учитывать отсутствие сведений в {source_field}',
        scope=f'Все клиенты; условие True: {rule}',
        n_rows=len(df_clients),
        value_before='Отдельный флаг отсутствовал',
        action='Создан флаг',
        value_after=f'Флаг рассчитан для {len(df_clients)} клиентов; True у {int(df_clients[flag].sum())}',
        rationale='Отдельно учитывать полноту сведений в групповых сравнениях и при выборе наблюдений для конкретного анализа. Исходные значения не изменяются и не заполняются.',
    )


Вывод: 
- сумма и источник дохода нужно проверить связь. 
- пропуски городов и сегментов получились при объединении одинаковый id клиентов

*месяц последнего входа тоже уже было объяснено раньше месте

## 1.7.1 Полнота сведений по сегментам
Сравним доли отсутствующих сведений внутри каждого сегмента

In [ ]:
def missing_by(group, flag):
    """Number of clients and proportion of missing values per group"""
    result = (
        df_clients.groupby(group, observed=True, dropna=False)[flag]
        .agg(n_clients='size', n_missing='sum', missing_pct='mean') # mean=sum/mean 
    )
    result['missing_pct'] = result['missing_pct'].mul(100)
    return result


segment_group = df_clients.groupby('segment', observed=True, dropna=False)

segment_counts = segment_group.size()
segment_rates = segment_group[flags].mean().mul(100)

segment_table = segment_rates.rename(columns=flag_labels).copy()
segment_table.insert(0, 'Клиентов', segment_counts)

display(segment_table.round(2))

# Подписи
heatmap_data = segment_rates.rename(columns=flag_labels).copy()
heatmap_data.index = [
    f'{segment if pd.notna(segment) else "Сегмент отсутствует"} '
    f'(n={int(n)})'
    for segment, n in zip(segment_rates.index, segment_counts)
]

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='Blues', vmin=0, vmax=100, linewidths=0.5, cbar_kws={'label': 'Отсутствует внутри группы, %'}, ax=ax)
ax.set(title='Полнота сведений по клиентским сегментам', xlabel='', ylabel='')
ax.tick_params(axis='both', labelrotation=0)
plt.tight_layout()
plt.show()


Вывод: видны некоторые закономерности, но они не объясняют наличие пропусков. В массовом сегменте клиенты больше всего скрывают информацию о доходах и реже остальных оставляли оценку.

## 1.7.2 Связь суммы дохода с наличием его источником
Проверим, как часто сумма дохода отсутсвует при известном и неизвестном источнике  

In [ ]:
source_presence = missing_by('income_source_missing', 'income_missing'
).reindex([False, True])

source_presence.index = [
    'Источник известен',
    'Источник неизвестен',
]

display(source_presence.round(2))

plot_data = source_presence.loc[
    source_presence['n_clients'].gt(0)
]

labels = [
    f'{name} (n={int(row["n_clients"])})'
    for name, row in plot_data.iterrows()
]

fig, ax = plt.subplots(figsize=(9, 3.5))

bars = ax.barh(labels, plot_data['missing_pct'])
ax.bar_label(bars, labels=[f'{value:.1f}%' for value in plot_data['missing_pct']], padding=4)
ax.set(title='Отсутствие суммы дохода при разной полноте источника', xlabel='Сумма дохода отсутствует, % группы', ylabel='', xlim=(0, max(5, plot_data['missing_pct'].max() * 1.3)))
ax.invert_yaxis()
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

unknown_source_with_income = (
    df_clients['income_source_missing']
    & ~df_clients['income_missing']
).sum()

print (f"У {unknown_source_with_income} клиентов источник неизвестен, но сумма дохода заполнена.")

Вывод: клиент, которые не указали источник дохода, пропускали сумму дохода в ~5 раз чаще, чем клиенты с зарплатой в другом банке или в нашем банке. 

## 1.7.3 Полнота оценок удовлетворённости 
Сравним доли отсутствующих оценок по уровням лояльности

In [ ]:
score_by_loyalty = missing_by('loyalty_tier', 'score_missing')

# Сортировка по доле пропусков облегчает сравнение
score_by_loyalty = score_by_loyalty.sort_values('missing_pct')

display(score_by_loyalty.round(2))

labels = [
    f'{name if pd.notna(name) else "Уровень отсутствует"} '
    f'(n={int(row["n_clients"])})'
    for name, row in score_by_loyalty.iterrows()
]

fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.barh(labels, score_by_loyalty['missing_pct'])
ax.bar_label(bars, labels=[f'{value:.1f}%' for value in score_by_loyalty['missing_pct']], padding=4)
ax.set(title='Отсутствие оценок по уровню лояльности', xlabel='Оценка отсутствует, % группы', ylabel='', xlim=(0, 105))
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

# Вычисляем разброс доли пропусков между группами лояльности для оценки неоднородности
score_range = (score_by_loyalty['missing_pct'].max() - score_by_loyalty['missing_pct'].min())


Вывод: Клиенты базового и серебряного уровней в ~2 раза реже оставляют оценки, чем клиенты золотого и платинового уровней.
Решение: satisfaction_score оставляем с пропусками

## 1.7.4 Совместные пропуски  и полнота анкет
Посчитаем, сколько из трёх исследуемых сведений
отсутствует у каждого клиента:

- сумма дохода;
- источник дохода;
- оценка.

Проверим, сколько клиентов останется,
если потребовать заполнения всех трёх полей

In [ ]:
n_missing = df_clients[flags].sum(axis=1)
complete_mask = n_missing.eq(0)

completeness = (
    n_missing.value_counts()
    .reindex(range(4), fill_value=0)
    .rename_axis('Отсутствует полей из трёх')
    .to_frame('Клиентов')
)

completeness['Доля базы, %'] = (
    completeness['Клиентов'].div(len(df_clients)).mul(100)
)

display(completeness.round(2))

# Оцениваем, как удаление неполных анкет исказит распределение клиентов по сегментам
segment_for_compare = (
    df_clients['segment']
    .astype('string')
    .fillna('Сегмент отсутствует')
)

all_counts = segment_for_compare.value_counts()
complete_counts = (
    segment_for_compare.loc[complete_mask]
    .value_counts()
    .reindex(all_counts.index, fill_value=0)
)

composition = pd.DataFrame({
    'Вся база, клиентов': all_counts,
    'Полные анкеты, клиентов': complete_counts,
})

composition['Вся база, %'] = (
    all_counts.div(len(df_clients)).mul(100)
)

n_complete = int(complete_mask.sum())

composition['Полные анкеты, %'] = (
    complete_counts.div(n_complete).mul(100)
    if n_complete > 0
    else np.nan
)

composition['Изменение доли, п.п.'] = (composition['Полные анкеты, %'] - composition['Вся база, %'])

display(composition.round(2))

print(f'Два или три пропуска: {n_missing.ge(2).sum()} ({n_missing.ge(2).mean():.2%})')
print(f'Все три сведения известны: {n_missing.eq(0).sum()} ({n_missing.eq(0).mean():.2%})')

# Фиксируем в журнале решение не удалять и не заполнять пропуски, чтобы не вносить искусственные искажения
retained_missing = {
    'monthly_income': ('income_missing', 'Неизвестный доход не равен нулю. Общее заполнение изменит распределение.'),
    'income_source': ('income_source_missing', 'Источник нельзя достоверно восстановить по наличию или размеру суммы.'),
    'satisfaction_score': ('score_missing', 'Заполнение создало бы ответы клиентов, которые оценку не предоставили.'),
}

for field, (flag, reason) in retained_missing.items():
    add_log(
        key=f'clients_keep_missing_{field}',
        field=field,
        issue='Отсутствуют содержательные сведения',
        scope=f'{flag} == True',
        n_rows=df_clients[flag].sum(),
        value_before='Пропуск или обозначение отсутствующих сведений',
        action='Оставлено без заполнения',
        value_after='Исходное отсутствие сведений сохранено',
        rationale=reason,
    )

add_log(
    key='clients_no_global_dropna',
    field='clients.*',
    issue='Неполные анкеты по сумме, источнику дохода или оценке',
    scope='Хотя бы один из трёх основных флагов равен True',
    n_rows=(~complete_mask).sum(),
    value_before='Анкеты с одним или несколькими отсутствующими полями',
    action='Оставлено; общее удаление неполных строк не применяется',
    value_after=f'В базе сохранены все {len(df_clients)} клиентов',
    rationale='Для каждого исследования нужны свои поля. Отсутствие оценки или дохода не должно автоматически исключать клиента из анализа операций.',
)

Вывод: Если требовать, чтобы все пропуски были заполнены, поэтому для каждого исследования надо выбирать только необходимые признаки

## Итоги
- клиентскую базу сохраняем без общего удаления неполных анкет
- доходы и оценки не заполняем нулями, медианами или средним
- для каждого расчёта подбираем наблюдения по наличию необходимых сведений 

In [ ]:
cleaning_log = pd.DataFrame(
    list(cleaning_log_entries.values()),
    columns=LOG_COLUMNS,
)

display(cleaning_log)

print('Решений в журнале:', len(cleaning_log))

In [ ]:
df_clients.columns.tolist()

## 2.  Аудит и подготовка таблицы transactions

## 2.1 Чтение и проверка типов
Читаем transactions.csv с разделителем «;» и кодировкой cp1251.


In [ ]:
# Загрузка транзакций с приведением типов и очисткой числовых значений
df_transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    sep=';',
    encoding='cp1251',
    dtype={
        'txn_id': 'str',
        'client_id': 'str',
        'currency': 'category',
        'category': 'category',
        'mcc_code': 'category',
        'channel': 'category',
        'status': 'category'
    },
    parse_dates=['txn_datetime'],
    date_format={'txn_datetime': 'ISO8601'},
    converters={
         # Обработка суммы: удаление пробелов, замена запятой на точку, защита от невалидных пропусков
        'amount': lambda value: float(
            ''.join(value.split()).replace(',', '.')
        ) if value.strip().casefold() not in {
            '', 'nan', 'na', 'n/a', 'null', 'none'
        } else np.nan
    }
)

n_transactions_loaded = len(df_transactions)

# Паспорт таблицы: типы, уникальные значения и пропуски
txn_overview = pd.DataFrame({
    'Тип': df_transactions.dtypes.astype(str),
    'Уникальных': df_transactions.nunique(),
    'Пропусков': df_transactions.isna().sum(),
    'Пропуски, %': df_transactions.isna().mean().mul(100),
})

print(
    f'Загружено {n_transactions_loaded:,} строк '
    f'× {df_transactions.shape[1]} столбцов'
)
display(txn_overview.round(2))
display(df_transactions.head(3))

# Сохраняем список исходных столбцов
txn_source_columns = df_transactions.columns.tolist()

# Фиксируем чтение и преобразование типов
add_log(
    key='transactions_read_types',
    field='transactions.*',
    issue='Нужны корректное чтение CSV и смысловые типы полей',
    scope='Все строки transactions.csv',
    n_rows=n_transactions_loaded,
    value_before='CSV: разделитель ;, кодировка cp1251, десятичная запятая',
    action='Заданы типы; преобразованы сумма и дата при чтении',
    value_after=(f'{len(df_transactions)} строк, {df_transactions.shape[1]} столбцов'),
    rationale=('Идентификаторы сохранены строками; сумма числовая; дата datetime. Строки не удалены.'),
)

Вывод: таблица transactions содержит 629'930 строк и 9 столбцов. Все столбцы прочитаны с нужными типами. Пропусков не обнаружено

## 2.2 Проверка категориальных значений и их пропусков

In [ ]:
# Список категориальных колонок транзакций для анализа и последующей нормализации
TXN_CAT_COLS = [ 'currency', 'category', 'mcc_code', 'channel', 'status']

# Смотрим исходные написания и частоты до нормализации
for col in TXN_CAT_COLS:
    counts = df_transactions[col].value_counts(dropna=False)

    category_overview = pd.DataFrame({
        'Количество': counts,
        'Доля, %': counts.div(len(df_transactions)).mul(100),
    })

    print(f'\nПоле: {col}')
    display(category_overview.round(2))

Вывод: Пропусков в категориальных столбцах нет, категориальные столбцы имеют ограниченные количества категорий. Выявлены разные варианты написания категорий в некоторых столбцах.
Решение: объединяем известные варианты написания

In [ ]:
normalization_rows = []

for col in TXN_CAT_COLS:
    before = df_transactions[col].astype('string')

    # Убираем внешние и повторные пробелы
    after = (
        before
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )

    # Пустая строка означает отсутствие значения
    after = after.mask(after.eq(''))

    if col == 'currency':
        after = after.str.upper()

    elif col == 'status':
        after = after.str.casefold()

    elif col == 'channel':
        after = after.str.capitalize().replace({
            'Сбп': 'СБП',
        })

    elif col == 'category':
        after = after.str.capitalize().replace({
            'Кафе/рестораны': 'Кафе и рестораны',
            'Связь и интернет': 'Связь и Интернет',
            'Азс': 'АЗС',
        })

    # Считаем изменения, не принимая два пропуска за различие
    changed = (
        before.ne(after).fillna(False)
        | before.isna().ne(after.isna())
    )

    df_transactions[col] = after.astype('category')

    normalization_rows.append({
        'Поле': col,
        'Вариантов до': before.nunique(),
        'Вариантов после': after.nunique(),
        'Изменено строк': int(changed.sum()),
        'Пропусков после': int(after.isna().sum()),
    })

    add_log(
        key=f'transactions_normalize_{col}',
        field=col,
        issue='Различия в написании категориальных значений',
        scope='Строки, изменившиеся при нормализации',
        n_rows=changed.sum(),
        value_before=f'{before.nunique()} заполненных вариантов',
        action='Нормализация написаний',
        value_after=f'{after.nunique()} заполненных вариантов',
        rationale=('Унифицированы пробелы, регистр и известные варианты названий. Пропуски не заполнены; смысл категорий и значения MCC не переопределяются.'),
    )

# Проверяем результат преобразований
normalization_summary = (
    pd.DataFrame(normalization_rows)
    .set_index('Поле')
)
display(normalization_summary)

print('\nКатегории операций после нормализации:')
display(
    df_transactions['category']
    .value_counts(dropna=False)
    .to_frame('Количество')
)

# Сохраняем таблицу после нормализации, до удаления повторов
transactions_normalized = df_transactions.copy(deep=True)

Вывод: категориальные значения приведены к единому виду

## 2.3 Проверка сумм, валют, статусов
Проверим пропуски, отрицательные суммы, нули

In [ ]:
amount = df_transactions['amount']

quality = pd.Series({
    'Всего операций': len(df_transactions),
    'Пропусков во всех полях': df_transactions.isna().sum().sum(),
    'Пропусков суммы': amount.isna().sum(),
    'Бесконечных сумм': np.isinf(
        amount.to_numpy(dtype=float, na_value=np.nan)
    ).sum(),
    'Нулевых сумм': amount.eq(0).sum(),
    'Отрицательных сумм': amount.lt(0).sum(),
    'Положительных сумм': amount.gt(0).sum(),
}, name='Количество')

display(quality.to_frame())

# Показываем примеры только при наличии проблемных сумм
amount_to_check = (
    amount.isna()
    | ~np.isfinite(amount)
    | amount.eq(0)
)

if amount_to_check.any():
    display(
        df_transactions.loc[
            amount_to_check,
            ['txn_id', 'amount', 'currency', 'status'],
        ].head(10)
    )

Вывод: пропусков, бесконечных сумм и нулевых значений нет, но есть отрицательные суммы, которые надо проверить, например, на согласованность знака суммы и статуса

In [ ]:
status = df_transactions['status'].astype('string')

# Невалидные суммы выделяем отдельно, а не считаем положительными
valid_amount = amount.notna() & np.isfinite(amount)

sign = pd.Series(
    'Не определён',
    index=df_transactions.index,
    name='Знак суммы',
)

sign.loc[valid_amount & amount.lt(0)] = 'Отрицательная'
sign.loc[valid_amount & amount.eq(0)] = 'Нулевая'
sign.loc[valid_amount & amount.gt(0)] = 'Положительная'

sign_counts = pd.crosstab(
    status.fillna('Статус отсутствует'),
    sign,
).reindex(
    columns=[ 'Отрицательная', 'Нулевая', 'Положительная', 'Не определён'],
    fill_value=0,
)

# Цвет показывает долю внутри статуса, подпись — число строк
sign_pct = (
    sign_counts
    .div(sign_counts.sum(axis=1), axis=0)
    .mul(100)
)

fig, ax = plt.subplots(figsize=(9, 3.5))

sns.heatmap(
    sign_pct,
    annot=sign_counts,
    fmt=',d',
    cmap='Blues',
    vmin=0,
    vmax=100,
    linewidths=0.5,
    cbar_kws={'label': 'Доля внутри статуса, %'},
    ax=ax,
)

ax.set(
    title='Согласованность знака суммы и статуса',
    xlabel='Знак суммы',
    ylabel='Статус',
)
ax.tick_params(axis='both', labelrotation=0)

plt.tight_layout()
plt.show()

# явный контроль противоречий и запись решения
expected_statuses = {'успешно', 'отклонено', 'возврат'}
unknown_status = status.isna() | ~status.isin(expected_statuses)
refund = status.eq('возврат').fillna(False)
non_refund = status.isin(['успешно', 'отклонено'])
sign_conflict = valid_amount & (
    (refund & amount.ge(0)) | (non_refund & amount.le(0))
)

transactions_amount_issues = df_transactions.loc[
    amount_to_check | unknown_status | sign_conflict
].copy()

add_log(
    key='transactions_keep_refunds', field='amount; status',
    issue='Отрицательные суммы требуют содержательной интерпретации',
    scope='Конечные отрицательные суммы со статусом возврат',
    n_rows=(valid_amount & amount.lt(0) & refund).sum(),
    value_before='Отрицательная сумма у возврата',
    action='Оставлено без изменения знака',
    value_after=(
    f'Возвратов с отрицательной конечной суммой сохранено: {int((valid_amount & amount.lt(0) & refund).sum())}; противоречий знака и известного статуса: {int(sign_conflict.sum())}; неизвестных или отсутствующих статусов: {int(unknown_status.sum())}; сумм, требующих проверки: {int(amount_to_check.sum())}'
),
    rationale=('Отрицательный знак возврата сохранён; абсолютное значение не подставляется. Диагностика предварительная, до удаления повторов. Возможные нарушения отдельно сохранены в transactions_amount_issues; эта запись не означает их автоматического исправления.'),
)

Вывод: отрицательные суммы это не ошибки! Возвраты сохраняют со знаком минус, успешные и отклонённые операции имеют положительную сумму.
Решение: возвраты сохраняем со знаком минус.

In [ ]:
# Для сравнения покупок не включаем возвраты и отклонённые попытки.
plot_data = df_transactions.loc[
    df_transactions['status'].astype('string')
    .str.strip().str.casefold().eq('успешно').fillna(False)
    & df_transactions['amount'].gt(0),
    ['currency', 'category', 'amount'],
].copy()

# Объединяем варианты написания категорий в выборке для графиков.
plot_data['category'] = (
    plot_data['category'].astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.capitalize()
    .replace({
        'Кафе/рестораны': 'Кафе и рестораны',
        'Азс': 'АЗС',
        'Связь и интернет': 'Связь и Интернет',
    })
    .fillna('Категория не указана')
)

for currency in ['RUB', 'USD', 'EUR']:
    data = plot_data.loc[plot_data['currency'].eq(currency)]

    if data.empty:
        continue

    # Сортируем категории по типичному размеру покупки.
    category_stats = (
        data.groupby('category', observed=True)['amount']
        .agg(n='size', median='median')
        .sort_values('median')
    )

    fig, ax = plt.subplots(figsize=(11, 6))

    sns.boxplot(
        data=data,
        x='amount',
        y='category',
        order=category_stats.index.tolist(),
        color='steelblue',
        whis=1.5,
        flierprops={
            'marker': 'o',
            'markersize': 3,
            'alpha': 0.35,
            'markerfacecolor': 'none',
        },
        ax=ax,
    )

    # Шкалу меняем после построения:
    # квартильные границы и усы рассчитаны по исходным суммам.
    ax.set_xscale('log')

    ax.set_yticks(
        range(len(category_stats)),
        labels=[
            f'{category} (n={int(n):,})'.replace(',', ' ')
            for category, n in category_stats['n'].items()
        ],
    )

    ax.set(
        title=f'Суммы успешных операций по категориям — {currency}',
        xlabel=f'Сумма, {currency} — логарифмическая шкала',
        ylabel='',
    )

    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)

    plt.tight_layout()
    plt.show()

## 2.4 Проверка уникальности операций
Рассмотрим столбец txn_id, который должен идентифицировать одну операцию.

In [ ]:
# Проверяем ключ до удаления повторов
txn_id = df_transactions['txn_id'].astype('string')

missing_txn_id = (
    txn_id.isna()
    | txn_id.str.strip().eq('').fillna(False)
)

n_before_duplicates = len(df_transactions)
n_unique_before = txn_id.nunique()

uniqueness_summary = pd.Series({
    'Всего строк': n_before_duplicates,
    'Уникальных заполненных txn_id': n_unique_before,
    'Пустых txn_id': missing_txn_id.sum(),
    'Лишних полных копий': df_transactions.duplicated().sum(),
    'Строк с повторяющимся заполненным txn_id': (
        ~missing_txn_id
        & txn_id.duplicated(keep=False)
    ).sum(),
}, name='Количество')

display(uniqueness_summary.to_frame())

# Для диагностики убираем полные копии только во временной таблице
transactions_no_copies = df_transactions.drop_duplicates()

conflict_id = transactions_no_copies['txn_id'].astype('string')
filled_id = (
    conflict_id.notna()
    & conflict_id.str.strip().ne('').fillna(False)
)

transaction_conflicts = (
    transactions_no_copies.loc[filled_id & conflict_id.duplicated(keep=False)]
    .sort_values('txn_id')
    .copy()
)

# Какие поля различаются у одного идентификатора
differing_fields = (
    transaction_conflicts
    .groupby('txn_id', observed=True)
    .nunique(dropna=False)
    .gt(1)
)

print(
    'Разных txn_id с конфликтом:',
    transaction_conflicts['txn_id'].nunique(),
)

display(
    differing_fields.sum()
    .loc[lambda values: values.gt(0)]
    .to_frame('Идентификаторов с различием')
)

# Показываем несколько групп целиком
example_ids = (
    transaction_conflicts['txn_id']
    .drop_duplicates()
    .head(3)
)

display(
    transaction_conflicts.loc[transaction_conflicts['txn_id'].isin(example_ids)]
)

Вывод: для данной таблицы полные копии не несут никакого смысла это просто ошибки, поэтому полные копии надо удалить, а основные не полные копии проверяем.
Решение: удалить полные копии, проверить конфиктующие пары

## 2.4.1 Удаляем полные копии


In [ ]:
# Сохраняем удаляемые копии для аудита
transaction_duplicates = df_transactions.loc[
    df_transactions.duplicated()
].copy()

df_transactions = df_transactions.drop_duplicates().copy()

print('Удалено полных копий:', len(transaction_duplicates))
print('Осталось строк:', len(df_transactions))

add_log(
    key='transactions_full_duplicates',
    field='transactions.*',
    issue='Полностью совпадающие строки, включая txn_id',
    scope='Полные копии после нормализации категорий',
    n_rows=len(transaction_duplicates),
    value_before=f'{n_before_duplicates} строк',
    action='Удалены полные копии',
    value_after=f'{len(df_transactions)} строк',
    rationale=( 'Одна операция не должна учитываться несколько раз. Сохранено первое вхождение; удалённые копии доступны в transaction_duplicates.'),
)

Вывод: удалены все полные копии

## 2.4.2 Разные версии времени операции
При одинаковых txn_id в строках отличается только время: в одной из строк отметка всегда 00:00:00 того же дня.
Проверим это точно и удалим строки конфликтующие строки со временем 00:00:00

In [ ]:
# обновляем диагностику после удаления полных копий
filled_id = (
    df_transactions['txn_id'].notna()
    & df_transactions['txn_id'].astype('string').str.strip().ne('').fillna(False)
)
transaction_conflicts = df_transactions.loc[
    filled_id & df_transactions['txn_id'].duplicated(keep=False)
].copy()

other_cols = [
    col for col in df_transactions.columns
    if col not in ['txn_id', 'txn_datetime']
]

# Совпадение всех полей, кроме времени
same_other_fields = (
    transaction_conflicts
    .groupby('txn_id', observed=True)[other_cols]
    .nunique(dropna=False)
    .le(1)
    .all(axis=1)
)

dates = transaction_conflicts['txn_datetime']

time_check = (
    transaction_conflicts
    .assign(
        calendar_date=dates.dt.normalize(),
        is_midnight=dates.notna() & dates.eq(dates.dt.normalize()),
    )
    .groupby('txn_id', observed=True)
    .agg(
        n_rows=('txn_datetime', 'size'),
        n_valid_dates=('txn_datetime', 'count'),
        n_days=('calendar_date', 'nunique'),
        n_midnight=('is_midnight', 'sum'),
    )
)

time_check['same_other_fields'] = same_other_fields

resolvable = (
    time_check['n_rows'].eq(2)
    & time_check['n_valid_dates'].eq(2)
    & time_check['n_days'].eq(1)
    & time_check['n_midnight'].eq(1)
    & time_check['same_other_fields']
)

resolvable_ids = time_check.index[resolvable]

display(pd.Series({
    'Конфликтующих txn_id': len(time_check),
    'Соответствуют правилу пары': resolvable.sum(),
    'Требуют отдельного разбора': (~resolvable).sum(),
}, name='Количество').to_frame())

if (~resolvable).any():
    display(time_check.loc[~resolvable].head(10))

# Удаляем полуночную версию только у проверенных пар:
# один txn_id, один день, остальные поля совпадают,
# ровно одна из двух записей имеет время 00:00:00
n_before = len(df_transactions)
dates_all = df_transactions['txn_datetime']

remove_midnight = (
    df_transactions['txn_id'].isin(resolvable_ids)
    & dates_all.notna()
    & dates_all.eq(dates_all.dt.normalize())
)

# Сохраняем удаляемые строки для аудита
removed_midnight_rows = df_transactions.loc[
    remove_midnight
].copy()

df_transactions = df_transactions.loc[
    ~remove_midnight
].copy()

n_removed = len(removed_midnight_rows)

# Повторно проверяем конфликты после удаления
transaction_conflicts = (
    df_transactions.loc[
        df_transactions['txn_id'].notna()
        & df_transactions['txn_id'].duplicated(keep=False)
    ]
    .sort_values(['txn_id', 'txn_datetime'])
    .copy()
)

# Итог обработки
display(pd.Series({
    'Строк до обработки': n_before,
    'Разрешено конфликтующих txn_id': (
        removed_midnight_rows['txn_id'].nunique()
    ),
    'Удалено полуночных строк': n_removed,
    'Строк после обработки': len(df_transactions),
    'Осталось конфликтующих txn_id': (
        transaction_conflicts['txn_id'].nunique()
    ),
    'Осталось конфликтующих строк': len(transaction_conflicts),
}, name='Количество').to_frame())

# Показываем сами записи, для которых правило не решило конфликт
if not transaction_conflicts.empty:
    print('\nНеразрешённые конфликтующие записи:')
    display(transaction_conflicts)
else:
    print('\nНеразрешённых конфликтов txn_id не осталось.')

# Фиксируем решение в существующем журнале
add_log(
    key='transactions_midnight_versions',
    field='txn_id; txn_datetime',
    issue='Разные версии времени одной операции',
    scope=('Две записи одного txn_id за один календарный день; остальные поля совпадают; ровно одна запись имеет время 00:00:00'
    ),
    n_rows=n_removed,
    value_before=(f'{n_before} строк; {len(resolvable_ids)} пар соответствуют правилу'),
    action='Удалена полуночная версия, сохранена версия с ненулевым временем',
    value_after=(f'{len(df_transactions)} строк; неразрешённых txn_id: {transaction_conflicts["txn_id"].nunique()}'
    ),
    rationale=('Принято допущение, что полуночная версия менее точна. Удаление применяется только к проверенным парам. Самостоятельные операции в полночь не удаляются. Удалённые строки сохранены в removed_midnight_rows; неразрешённые записи — в transaction_conflicts.'),
)

Вывод: все конфликтующие строки отличаются только временем. У всех дубликатов есть строчка со временем реальной покупки и временем 00:00:00, поэтому дубликаты с 00:00:00 удалили

## 2.5 Проверка дат(периода)
Исследуемый период с 1 января 2024 до 1 июля 2025

In [ ]:
# Даты преобразованы при загрузке данных.
period_start = pd.Timestamp('2024-01-01')
period_end_exclusive = pd.Timestamp('2025-07-01')

dates = df_transactions['txn_datetime']

missing_date = dates.isna()
before_period = dates.lt(period_start)
after_period = dates.ge(period_end_exclusive)

in_period = (
    dates.notna()
    & dates.ge(period_start)
    & dates.lt(period_end_exclusive)
)

period_summary = pd.Series({
    'Всего операций': len(df_transactions),
    'Без даты': missing_date.sum(),
    'Раньше 01.01.2024': before_period.sum(),
    'Начиная с 01.07.2025': after_period.sum(),
    'В исследуемом периоде': in_period.sum(),
}, name='Количество')

display(period_summary.to_frame())

print('Минимальная дата:', dates.min())
print('Максимальная дата:', dates.max())

# Отдельные выборки: исходную таблицу не перезаписываем.
transactions_period = df_transactions.loc[in_period].copy()

transactions_outside_period = df_transactions.loc[
    before_period | after_period
].copy()

transactions_without_date = df_transactions.loc[missing_date].copy()

if not transactions_outside_period.empty:
    print('\nПримеры операций вне периода:')
    display(
        transactions_outside_period[
            ['txn_id', 'client_id', 'txn_datetime',
             'amount', 'currency', 'status']
        ].sort_values('txn_datetime').head(10)
    )



add_log(
    key='transactions_period_selection', field='txn_datetime',
    issue='Часть операций не относится к исследуемому периоду',
    scope='Все операции после обработки повторов',
    n_rows=len(transactions_outside_period) + len(transactions_without_date),
    value_before=f'{len(df_transactions)} операций всех дат',
    action='Создана отдельная выборка периода; исходная очищенная таблица сохранена',
    value_after=f'{len(transactions_period)} в периоде; {len(transactions_outside_period)} вне периода; {len(transactions_without_date)} без даты',
    rationale='Границы [2024-01-01, 2025-07-01); внепериодные и недатированные записи сохранены отдельно.',
)

Вывод: есть 3 операции предшествующая мсследуемому периоду и 78 операций после исследуемого периода. Нельзя использовать операции не из исследуемого периода.
Решение: внепериодные операции сохраняем в transactions_outside_period,
но не используем для исследования заданного периода

## 2.6 Проверим согласованность дат и наличие клиента в справочнике clients
Проверим соответствие даты транзакции и открытия карты(регистрации), для всех ли операций найден клиент.


In [ ]:
client_date_cols = ['registration_date', 'card_open_date']

# Присоединяем только поля, необходимые для проверки
transactions_dates_check = transactions_period.merge(
    df_clients[['client_id', *client_date_cols]],
    on='client_id',
    how='left',
    validate='many_to_one',
    indicator='client_match',
)

# Определяем наличие клиентской анкеты
matched = transactions_dates_check['client_match'].eq('both')
no_client = ~matched

# Сравниваем календарные дни, не учитывая время внутри дня
txn_day = transactions_dates_check['txn_datetime'].dt.normalize()

registration_day = (
    transactions_dates_check['registration_date'].dt.normalize()
)

card_open_day = (
    transactions_dates_check['card_open_date'].dt.normalize()
)

# Проверка доступна, если найдена анкета и заполнены обе даты
date_checkable_registration = (
    matched
    & txn_day.notna()
    & registration_day.notna()
)

date_checkable_card = (
    matched
    & txn_day.notna()
    & card_open_day.notna()
)

# Отмечаем операции раньше даты регистрации
transactions_dates_check['before_registration'] = (
    date_checkable_registration
    & txn_day.lt(registration_day)
)

# Отмечаем операции раньше даты открытия карты
transactions_dates_check['before_card_open'] = (
    date_checkable_card
    & txn_day.lt(card_open_day)
)

# Сохраняем операции без найденной клиентской анкеты
transactions_without_client = transactions_dates_check.loc[
    no_client
].copy()

# Сохраняем операции хотя бы с одним временным противоречием
date_conflicts = transactions_dates_check.loc[
    transactions_dates_check[
        ['before_registration', 'before_card_open']
    ].any(axis=1)
].copy()

# Фиксируем диагностику после создания всех признаков и выборок
add_log(
    key='transactions_date_diagnostic_flags',
    field='before_registration; before_card_open',
    issue=('Нужно проверить согласованность дат операций и клиентских событий'),
    scope=('Операции периода в диагностической таблице transactions_dates_check'),
    n_rows=len(transactions_dates_check),
    value_before='Диагностические признаки отсутствовали',
    action=('Созданы признаки проверки дат и сохранены временные противоречия'),
    value_after=(f'До регистрации: {int(transactions_dates_check["before_registration"].sum())}; до открытия карты: {int(transactions_dates_check["before_card_open"].sum())}; операций хотя бы с одним нарушением: {len(date_conflicts)}; доступна проверка регистрации: {int(date_checkable_registration.sum())}; доступна проверка открытия карты: {int(date_checkable_card.sum())}; без анкеты: {int(no_client.sum())}'),
    rationale=('Сравниваются календарные дни. Признаки созданы только в диагностической таблице; рабочие операции не изменены. False не доказывает согласованность, если нет анкеты или необходимой даты. Результат относится к этапу до последующих исключений.'),
)

# Сводка результатов диагностики
date_summary = pd.DataFrame({
    'Количество': [
        len(transactions_dates_check),
        no_client.sum(),
        transactions_dates_check.loc[
            no_client, 'client_id'
        ].nunique(),
        (matched & registration_day.isna()).sum(),
        (matched & card_open_day.isna()).sum(),
        date_checkable_registration.sum(),
        date_checkable_card.sum(),
        transactions_dates_check['before_registration'].sum(),
        transactions_dates_check['before_card_open'].sum(),
        len(date_conflicts),
    ],
}, index=[
    'Всего операций в диагностической таблице',
    'Операций без клиента в очищенном справочнике',
    'Несопоставленных заполненных client_id',
    'Сопоставленных операций без даты регистрации',
    'Сопоставленных операций без даты открытия карты',
    'Доступна проверка относительно регистрации',
    'Доступна проверка относительно открытия карты',
    'Операций до регистрации',
    'Операций до открытия карты',
    'Операций хотя бы с одним временным противоречием',
])

# Для операционных показателей считаем долю от всех строк таблицы
total_ops = len(transactions_dates_check)

date_summary['Доля от всех операций, %'] = (
    date_summary['Количество']
    .div(total_ops if total_ops else np.nan)
    .mul(100)
    .round(4)
)

# Число уникальных клиентов не делим на количество операций
date_summary.loc[
    'Несопоставленных заполненных client_id',
    'Доля от всех операций, %',
] = np.nan

display(date_summary)

# Показываем примеры операций без анкеты и их распределение по MCC
if not transactions_without_client.empty:
    print('\nПримеры операций без сопоставленного клиента:')

    display(
        transactions_without_client[
            ['txn_id','client_id','txn_datetime','status','channel','mcc_code',]
        ].head(10)
    )

    print('\nРаспределение операций без клиента по mcc_code (всего: {len(transactions_without_client)}):')

    mcc_summary = (
        transactions_without_client['mcc_code']
        .value_counts(dropna=False)
        .to_frame('Количество операций')
    )

    mcc_summary['Доля, %'] = (
        mcc_summary['Количество операций']
        .div(len(transactions_without_client))
        .mul(100)
        .round(2)
    )

    display(mcc_summary)

# Показываем примеры обнаруженных временных противоречий
if not date_conflicts.empty:
    print('\nПримеры временных несогласованностей:')

    display(
        date_conflicts[
            ['txn_id','client_id','txn_datetime','registration_date','card_open_date','channel','status','category','before_registration','before_card_open']
        ]
        .sort_values(['client_id', 'txn_datetime'])
        .head(10)
    )

Вывод: 
- для 26 583 операций — 4,23% выборки — не найдены анкеты
в очищенном клиентском справочнике.
- найдена 1 операция до открытия карты

Решение: 
- операции без клиентской анкеты не удаляем, а сохраняем в transactions_without_client и продолжаем исследование в 2.7. Отсутсвие клиента не говорит, что операция неверная
- операция до открытия карты не может быть выполнена, поэтому удаляем её

In [ ]:
# Выбираем карточные операции до указанной даты открытия карты
mask_to_drop = (
    date_conflicts['before_card_open']
    & date_conflicts['channel'].eq('Карта').fillna(False)
)

# Сохраняем записи с клиентскими датами и результатами проверки
rows_to_drop = date_conflicts.loc[mask_to_drop].copy()
txn_ids_to_drop = rows_to_drop['txn_id'].unique()

# Считаем фактические строки для удаления в каждой таблице
remove_from_main = df_transactions['txn_id'].isin(txn_ids_to_drop)
remove_from_period = transactions_period['txn_id'].isin(txn_ids_to_drop)

n_main_before = len(df_transactions)
n_period_before = len(transactions_period)

n_removed_main = int(remove_from_main.sum())
n_removed_period = int(remove_from_period.sum())

# Удаляем из общей рабочей таблицы и выборки периода
df_transactions = df_transactions.loc[
    ~remove_from_main
].copy()

transactions_period = transactions_period.loc[
    ~remove_from_period
].copy()

# Заменяет прежнюю запись об исключении только из выборки периода
add_log(
    key='transactions_before_card_exclusion',
    field='txn_datetime; card_open_date; channel',
    issue='Карточная операция раньше указанной даты открытия карты',
    scope='before_card_open == True и channel == Карта',
    n_rows=n_removed_main,
    value_before=(f'df_transactions: {n_main_before} строк; transactions_period: {n_period_before} строк'
    ),
    action='Удалено из рабочей таблицы и выборки периода',
    value_after=(f'df_transactions: {len(df_transactions)} строк; transactions_period: {len(transactions_period)} строк; в rows_to_drop сохранено {len(rows_to_drop)} записей'),
rationale=('Применено принятое в анализе правило: карточные операции раньше указанной даты открытия карты исключаются. Правило предполагает, что card_open_date относится к карте, использованной в операции. Какое именно исходное поле ошибочно, не установлено. Удалённые записи сохранены в rows_to_drop; исходный CSV не изменён. n_rows — число удалённых строк общей рабочей таблицы; уникальность txn_id проверена. Удаление тех же операций из периодной копии повторно не суммируется.'),
)

print('Удалено операций:', n_removed_main)
print('Осталось в df_transactions:', len(df_transactions))
print('Осталось в transactions_period:', len(transactions_period))

if not rows_to_drop.empty:
    print('\nУдалённые операции, сохранённые для аудита:')

    display(
        rows_to_drop[
            ['txn_id', 'client_id', 'txn_datetime','registration_date', 'card_open_date','channel', 'status', 'category',]
        ]
    )

Вывод: удалена одна карточная операция, выполненная раньше указанной даты открытия карты. Копия сохранена в rows_to_drop

Вывод: доля операций без сопоставленного клиент за месяц уменьшается. Однозначную причину такого поведения нельзя определить, но возможно система потеряла старых клиентов или они ушли из банка и их удалили из таблицы клиентов.

## 2.7 Полнота связи с клиентами и итоговые аналитические выборки

## 2.7.1 Операции без клиентской анкеты
Проверим есть ли операции для которых не найден клиент, сравним их долю.

In [ ]:
# Создаём свежую диагностическую выборку
link_analysis = transactions_period.copy()

link_analysis['client_missing'] = (
    ~link_analysis['client_id'].isin(df_clients['client_id'])
)

# Фиксируем создание диагностического флага
add_log(
    key='transactions_diagnostic_client_missing',
    field='client_missing',
    issue='Нужно выделить операции без клиентской анкеты',
    scope='Все операции диагностической таблицы link_analysis',
    n_rows=len(link_analysis),
    value_before='Диагностический флаг отсутствовал',
    action='Создан флаг в отдельной таблице',
    value_after=(f'True у {int(link_analysis["client_missing"].sum())} операций'),
    rationale=('client_missing показывает отсутствие client_id в очищенном клиентском справочнике. Рабочие таблицы не изменены.'),
)

unknown_transactions = link_analysis.loc[
    link_analysis['client_missing']
].copy()

overall_rate = (link_analysis['client_missing'].sum() / len(link_analysis) * 100)

month = link_analysis['txn_datetime'].dt.to_period('M')

monthly_link = (
    link_analysis
    .groupby(month)['client_missing']
    .agg(total='size', missing='sum')
    .reindex(pd.period_range('2024-01', '2025-06', freq='M'))
)

monthly_link[['total', 'missing']] = (
    monthly_link[['total', 'missing']].fillna(0).astype(int)
)

monthly_link['missing_pct'] = (
    monthly_link['missing']
    .div(monthly_link['total'].replace(0, float('nan')))
    .mul(100)
)

fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(
    monthly_link.index.astype(str),
    monthly_link['missing_pct'],
    marker='o',
    color='steelblue',
)

ax.axhline(
    overall_rate,
    color='darkorange',
    linestyle='--',
    label=f'Весь период: {overall_rate:.2f}%',
)

ax.set(
    title='Доля операций без сопоставленного клиента по месяцам',
    xlabel='Месяц',
    ylabel='Доля операций, %',
)

ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.2)
ax.legend()

sns.despine()
plt.tight_layout()
plt.show()

display(monthly_link.round(2))

Вывод: доля операций без клиентской анкеты снижается. Причина отсутствия анкет по этим данным не установлена.


In [ ]:
# Разделяем операции по наличию клиентской анкеты
client_found = transactions_period['client_id'].isin(
    df_clients['client_id']
)

transactions_linked = transactions_period.loc[
    client_found
].copy()

transactions_unlinked = transactions_period.loc[
    ~client_found
].copy()


# Отдельно считаем операции и разные client_id
link_summary = pd.DataFrame({
    'Операций': [
        len(transactions_period),
        len(transactions_linked),
        len(transactions_unlinked),
    ],
    'Уникальных заполненных client_id': [
        transactions_period['client_id'].nunique(),
        transactions_linked['client_id'].nunique(),
        transactions_unlinked['client_id'].nunique(),
    ],
}, index=[
    'Все операции периода',
    'Анкета найдена',
    'Анкета не найдена',
])

link_summary['Доля операций, %'] = (
    link_summary['Операций']
    .div(len(transactions_period) if len(transactions_period) else np.nan)
    .mul(100)
)

display(link_summary.round(2))

# количество операций на каждый отсутствующий в справочнике ID
unlinked_clients_summary = (
    transactions_unlinked
    .groupby('client_id', dropna=False)
    .agg(
        n_operations=('txn_id', 'size'),
        first_operation=('txn_datetime', 'min'),
        last_operation=('txn_datetime', 'max'),
    )
    .sort_values(
        ['n_operations', 'first_operation'],
        ascending=[False, True],
    )
    .rename(columns={
        'n_operations': 'Количество операций',
        'first_operation': 'Первая операция',
        'last_operation': 'Последняя операция',
    })
)


print('В справочнике не найдены анкеты для {transactions_unlinked["client_id"].nunique()} различных заполненных client_id.'
)

print('Количество операций без анкеты: {len(transactions_unlinked):,}'
)


# Фиксируем разделение по доступности клиентских данных
add_log(
    key='transactions_client_link_selection',
    field='client_id',
    issue='Для части операций не найдена клиентская анкета',
    scope='Операции периода после проверки дат',
    n_rows=len(transactions_unlinked),
    value_before=f'{len(transactions_period)} операций периода',
    action='Созданы отдельные выборки; операции не удалены',
    value_after=(f'{len(transactions_linked)} операций с анкетой; {len(transactions_unlinked)} без анкеты; {transactions_unlinked["client_id"].nunique()} несопоставленных заполненных client_id'),
    rationale=('Все операции сохраняются для общего анализа. Для связей с клиентскими характеристиками используется transactions_linked.'),
)

Вывод: для 26'583 (4.23%) оперций не найдены 353 клиента
Решение:
 - не удаляем из transactions_period
 - для анализа клиентских характеристик исспользуем transactions_linked.


## 2.7.2. Распределение сумм успешных операций

Исследуем суммы успешных покупок отдельно по валютам.
Возвраты и отклонённые попытки в эту выборку не включаем.

In [ ]:
# Выбираем успешные операции с положительной конечной суммой
successful = transactions_period.loc[
    transactions_period['status']
    .astype('string')
    .eq('успешно')
    .fillna(False)
    & transactions_period['amount'].gt(0)
    & np.isfinite(transactions_period['amount'])
].copy()

# список валют больше не зависит от amount_stats
currencies = [
    currency
    for currency in ['RUB', 'USD', 'EUR']
    if successful['currency'].eq(currency).any()
]

add_log(
    key='transactions_successful_selection',
    field='status; amount',
    issue='Для распределения покупок нужна отдельная выборка',
    scope='Операции исследуемого периода после очистки',
    n_rows=len(transactions_period) - len(successful),
    value_before=f'{len(transactions_period)} операций всех статусов',
    action='Создана отдельная выборка successful',
    value_after=f'{len(successful)} успешных операций для анализа сумм',
    rationale=('Выбраны положительные конечные суммы со статусом успешно. Возвраты и отклонённые попытки остаются в transactions_period. Валюты анализируются отдельно.'),
)

In [ ]:
from scipy import stats

if currencies:
    fig, axes = plt.subplots(len(currencies), 3, figsize=(16, 3.8 * len(currencies)), squeeze=False)
    shape_rows = []
    for row, currency in enumerate(currencies):
        values = successful.loc[successful['currency'].eq(currency), 'amount']
        p99 = values.quantile(0.99)
        visible = values.loc[values.le(p99)]
        log_values = np.log(values)
        ax_main, ax_log, ax_qq = axes[row]
        # Основная масса сумм без верхнего хвоста — только для графика
        sns.histplot(x=visible, bins=45, ax=ax_main, color='steelblue')
        # Среднее и медиана 
        ax_main.axvline(values.mean(),color='crimson',linestyle='-.',linewidth=1.5,label=f'Среднее: {values.mean():,.2f}')
        ax_main.axvline(values.median(),color='darkorange',linestyle='--',linewidth=1.5,label=f'Медиана: {values.median():,.2f}')
        ax_main.legend(fontsize=8)
        ax_main.set(title=f'{currency}: до P99; скрыто {int(values.gt(p99).sum())}',
                    xlabel=f'Сумма, {currency}', ylabel='Количество')
        
         # Все суммы в логарифмической шкале
        sns.histplot(x=log_values, bins=45, ax=ax_log, color='steelblue')
        # Среднее и медиана 
        ax_log.axvline(log_values.mean(),color='crimson',linestyle='-.',linewidth=1.5,label=f'Среднее: {log_values.mean():,.2f}')
        ax_log.axvline(log_values.median(),color='darkorange',linestyle='--',linewidth=1.5,label=f'Медиана: {log_values.median():,.2f}')
        ax_log.legend(fontsize=8)
        ax_log.set(title=f'{currency}: все ln(суммы), n={len(values)}',
                   xlabel='ln(суммы)', ylabel='Количество')
        
        # Сравнение логарифмов сумм с нормальным распределением
        if len(values) >= 3 and values.nunique() > 1:
            (theoretical, observed), (slope, intercept, _) = stats.probplot(log_values, dist='norm')
            ax_qq.scatter(theoretical, observed, s=7, alpha=0.35)
            ax_qq.plot(theoretical, slope * theoretical + intercept, color='darkorange')
        ax_qq.set(title=f'{currency}: Q–Q ln(суммы)', xlabel='Квантили N(0, 1)', ylabel='Квантили ln(суммы)')
        shape_rows.append({'Валюта': currency, 'Асимметрия суммы': values.skew(),
                           'Асимметрия ln': log_values.skew(), 'Эксцесс ln': log_values.kurt()})
        for ax in axes[row]:
            sns.despine(ax=ax)
    plt.tight_layout()
    
    shape_summary = (
        pd.DataFrame(shape_rows)
        .set_index('Валюта')
    )
    
    plt.show()
    display(pd.DataFrame(shape_rows).set_index('Валюта').round(3))

Вывод: 

## 2.8 Согласованность категории операций и МСС 
Проверим, какие категории чаще всего встречаются
при каждом значении mcc_code.

In [ ]:
# Создаём отдельную таблицу для диагностики
category_check = transactions_period.copy()

# Приводим поля к строкам для проверки заполненности
mcc_text = (
    category_check['mcc_code']
    .astype('string')
    .str.strip()
)

category_text = (
    category_check['category']
    .astype('string')
    .str.strip()
)

valid_pair = (
    mcc_text.notna()
    & mcc_text.ne('').fillna(False)
    & category_text.notna()
    & category_text.ne('').fillna(False)
)

print('Операций без заполненной пары MCC–категория:',int((~valid_pair).sum()),)

# Сохраняем неполные пары отдельно, не удаляя их из рабочей таблицы
category_missing_pairs = category_check.loc[~valid_pair].copy()

category_check = category_check.loc[valid_pair].copy()
category_check['mcc_code'] = mcc_text.loc[valid_pair]
category_check['category'] = category_text.loc[valid_pair]

# Считаем частоты сочетаний
counts = pd.crosstab(
    category_check['mcc_code'],
    category_check['category'],
)

# Основную категорию назначаем только при единственном максимуме
max_count = counts.max(axis=1)
unique_leader = counts.eq(max_count, axis=0).sum(axis=1).eq(1)
main_category = counts.idxmax(axis=1).where(unique_leader)

mcc_category_summary = pd.DataFrame({
    'Основная категория': main_category,
    'Всего операций': counts.sum(axis=1),
    'Операций в основной категории': max_count.where(unique_leader),
})

mcc_category_summary['Доля основной категории, %'] = (
    mcc_category_summary['Операций в основной категории']
    .div(mcc_category_summary['Всего операций'])
    .mul(100)
)

mcc_category_summary['Других операций'] = (
    mcc_category_summary['Всего операций']
    - mcc_category_summary['Операций в основной категории']
)

mcc_category_summary['Доля других операций, %'] = (
    mcc_category_summary['Других операций']
    .div(mcc_category_summary['Всего операций'])
    .mul(100)
)

display(mcc_category_summary.round(2))

# Сохраняем ожидаемое по частоте соответствие только для диагностики
category_check['category_by_mcc_mode'] = (
    category_check['mcc_code'].map(main_category)
)

assessed = category_check['category_by_mcc_mode'].notna()

# При неопределённой основной категории оставляем NA,
# а не записываем False — такую операцию мы не оценили
category_check['category_differs'] = (
    category_check['category']
    .ne(category_check['category_by_mcc_mode'])
    .astype('boolean')
    .where(assessed)
)

category_mismatches = category_check.loc[
    category_check['category_differs'].fillna(False)
].copy()

print('Операций с определённой основной категорией:', int(assessed.sum()))
print('Операций без однозначного соответствия:', int((~assessed).sum()))
print('Расхождений с основной категорией:', len(category_mismatches))


add_log(
    key='transactions_mcc_category_diagnostics',
    field='mcc_code; category; category_by_mcc_mode; category_differs',
    issue='Категория операции может отличаться от наиболее частой категории MCC',
    scope='Диагностика операций периода с заполненной парой MCC–категория',
    n_rows=len(category_check),
    value_before='Диагностическое соответствие MCC–категория отсутствовало',
    action=('Созданы диагностические признаки; исходные категории сохранены'
    ),
    value_after=(f'Оценено операций: {int(assessed.sum())}; расхождений с модальной категорией: {len(category_mismatches)}; без однозначной модальной категории: {int((~assessed).sum())}; без заполненной пары: {len(category_missing_pairs)}'),
    rationale=('Основная категория определена по единственному максимуму частоты внутри MCC в исследуемой выборке. Это эмпирический ориентир, а не внешний справочник. Расхождение не доказывает ошибку, поэтому category не исправляется и операции не удаляются. При отсутствии единственного максимума category_differs оставлен неопределённым. Неполные пары сохранены отдельно.'),
)


if not category_mismatches.empty:
    print('\nПримеры расхождений:')

    display(
        category_mismatches[
            ['txn_id', 'client_id', 'txn_datetime','mcc_code', 'category', 'category_by_mcc_mode','channel', 'status',]
        ].head(10)
    )

Вывод: у каждого из 9 МСС имеется одна преобладающая категория, её доля порядка 99%, и 1673 операции с другими категориями. 
Решение: сохраняем расхождения в category_mismatches.

## 2.8.1 Расхождения по месяцам
Проверим, сосредоточены ли расхождения в отдельных месяцах.

In [ ]:
# Оставляем операции с однозначно определённой основной категорией
valid_mcc_data = category_check.loc[assessed].copy()

# Рассчитываем общую долю отклонений за весь период
overall_mismatch_rate = valid_mcc_data['category_differs'].mean() * 100

# Группировка по месяцам
monthly_category = (
    valid_mcc_data
    .groupby(valid_mcc_data['txn_datetime'].dt.to_period('M'))['category_differs']
    .agg(total='size', differing='sum')
    .reindex(pd.period_range('2024-01', '2025-06', freq='M'))
)

monthly_category[['total', 'differing']] = (
    monthly_category[['total', 'differing']]
    .fillna(0)
    .astype(int)
)

monthly_category['share_pct'] = (
    monthly_category['differing']
    .div(monthly_category['total'].replace(0, float('nan')))
    .mul(100)
)

# Построение графика
fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(
    monthly_category.index.astype(str),
    monthly_category['share_pct'],
    marker='o',
    label='Доля за месяц',
)

ax.axhline(
    overall_mismatch_rate,  # Используем рассчитанную общую долю
    color='darkorange',
    linestyle='--',
    label=f'Весь период: {overall_mismatch_rate:.2f}%',
)

# Нулевое начало оси не преувеличивает небольшие колебания.
ax.set_ylim(0, max(monthly_category['share_pct'].max(), overall_mismatch_rate) * 1.2,)

ax.set(
    title='Расхождения категории с основным соответствием MCC',
    xlabel='Месяц',
    ylabel='Доля отклонений, %',
)

ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.2)
ax.legend()

sns.despine()
plt.tight_layout()
plt.show()

Вывод: резких скачков доли расхождения по месяцам не наблюдается, наблюдаётся очень маленькое уменьшение доли отклонений. Проблема не ограничивается
одним отдельным месяцем.
Решение: операции не удаляем и не меняем

In [ ]:
df_transactions.columns.tolist()

### Уточнить можно ли use scipy!!!

In [ ]:
from scipy import stats

# Сохраняем реальное расстояние между месяцами,
# даже если в каком-то месяце нет операций.
trend_data = monthly_category.sort_index().copy()

trend_data['month_number'] = (
    trend_data.index.year * 12 + trend_data.index.month
)
trend_data['month_number'] -= trend_data['month_number'].min()

trend_data = trend_data.loc[
    trend_data['total'].gt(0)
    & trend_data['share_pct'].notna()
].copy()

x = trend_data['month_number'].to_numpy(dtype=float)
y = trend_data['share_pct'].to_numpy(dtype=float)

# Двусторонняя проверка: не выбираем направление теста
# только потому, что уже увидели снижение на графике.
reg = stats.linregress(x, y)

# 95%-й доверительный интервал наклона.
critical_t = stats.t.ppf(0.975, df=len(trend_data) - 2)

slope_low = reg.slope - critical_t * reg.stderr
slope_high = reg.slope + critical_t * reg.stderr

trend_data['trend_pct'] = reg.intercept + reg.slope * x

# Изменение по линии между первым и последним доступными месяцами.
fitted_change = reg.slope * (x.max() - x.min())

trend_result = pd.Series({
    'Месяцев в анализе': len(trend_data),
    'Наклон, п.п. в месяц': reg.slope,
    'Нижняя граница 95% ДИ наклона': slope_low,
    'Верхняя граница 95% ДИ наклона': slope_high,
    'Изменение по линии за период, п.п.': fitted_change,
    'R²': reg.rvalue ** 2,
    'p-value': reg.pvalue,
}, name='Результат')

display(trend_result.to_frame().round(6))

fig, ax = plt.subplots(figsize=(11, 4))

labels = trend_data.index.astype(str)

ax.plot(
    labels,
    trend_data['share_pct'],
    marker='o',
    label='Фактическая доля',
)

ax.plot(
    labels,
    trend_data['trend_pct'],
    color='darkorange',
    linestyle='--',
    label=f'Линейный тренд: {reg.slope:+.4f} п.п./месяц',
)

ax.set(
    title='Временной тренд расхождений категории и MCC',
    xlabel='Месяц',
    ylabel='Доля отклонений, %',
)

ax.set_ylim(0, max( trend_data['share_pct'].max(), trend_data['trend_pct'].max(),) * 1.2,)

ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.2)
ax.legend()

sns.despine()
plt.tight_layout()
plt.show()

Вывод:регрессия описала небольшие снижения доли отклонений.

In [ ]:
from pathlib import Path

# Собираем решения всех глав в итоговую таблицу
cleaning_log = pd.DataFrame(
    list(cleaning_log_entries.values()),
    columns=LOG_COLUMNS,
)

# Сохраняем в текущую рабочую папку
log_path = Path('cleaning_log.csv')

cleaning_log.to_csv(
    log_path,
    index=False,
    encoding='utf-8-sig',
)

print(f'Журнал сохранён: {log_path.resolve()}')
print(f'Количество решений: {len(cleaning_log)}')

display(cleaning_log)

## Глава 3. Метрика и расчёт клиентской активности
Цель - определить активность и получить показатели, по которым оценивать изменения на уровне клиента
## 3.1 Определение метрики
Основная метрика - среднее количество успешных оплат за месяц

In [ ]:
BASE_START = pd.Timestamp('2024-01-01')
CURRENT_START = pd.Timestamp('2025-01-01')
ACTIVITY_END = pd.Timestamp('2025-07-01')
ACTIVITY_MONTHS = pd.period_range('2024-01', '2025-06', freq='M')

period_ok = (
    transactions_period['txn_datetime'].ge(BASE_START)
    & transactions_period['txn_datetime'].lt(ACTIVITY_END)
)

# Справочник сохраняет все исходные клиентские признаки и флаги
activity_clients = df_clients.copy()
status_activity = transactions_period['status'].astype('string')
payment_mask = (
    status_activity.eq('успешно').fillna(False)
    & transactions_period['amount'].gt(0)
    & np.isfinite(transactions_period['amount'])
)
activity_payments_all = transactions_period.loc[payment_mask].copy()
linked_mask = activity_payments_all['client_id'].isin(activity_clients['client_id'])
activity_payments = activity_payments_all.loc[linked_mask].copy()
# Несопоставленные оплаты сохраняем 
activity_payments_unlinked = activity_payments_all.loc[~linked_mask].copy()
activity_payments['month'] = activity_payments['txn_datetime'].dt.to_period('M')

display(pd.Series({
    'Операций периода всех статусов': len(transactions_period),
    'Успешных оплат': len(activity_payments_all),
    'Успешных оплат с анкетой': len(activity_payments),
    'Успешных оплат без анкеты': len(activity_payments_unlinked),
}, name='Количество').to_frame())

Решение: для анализа клиентов будем использовать только оплаты с найденным клиентов в clients. Несопоставленные оплаты сохраняем в activity_payments_unlinked.

## 3.2 Доступность истории
Берём клиентов зарегистрированных в банке в исследуемый период

In [ ]:
registration = activity_clients['registration_date']
activity_clients['activity_history_group'] = np.select(
    [registration.isna(), registration.ge(ACTIVITY_END), registration.gt(BASE_START)],
    ['Дата регистрации неизвестна', 'Регистрация после периода', 'Нет полной базовой истории'],
    default='Полная история для сравнения',
)
activity_clients['activity_comparable'] = registration.notna() & registration.le(BASE_START)

history_summary = activity_clients['activity_history_group'].value_counts().to_frame('Клиентов')
history_summary['Доля базы, %'] = history_summary['Клиентов'].div(len(activity_clients)).mul(100)
display(history_summary.round(2))

# Не смешиваем реальные нули с отсутствующей историей
registration_by_id = activity_clients.set_index('client_id')['registration_date']
payment_registration = activity_payments['client_id'].map(registration_by_id)
activity_pre_registration = activity_payments.loc[
    payment_registration.notna()
    & activity_payments['txn_datetime'].dt.normalize().lt(payment_registration.dt.normalize())
].copy()


Вывод: для сравнения и анлиза используем 5'542 клиента

## 3.3 Помесячная таблица
Соберём таблицу для каждого клиента по каждому месяцу с количеством операций с учётом даты регистрации

In [ ]:
# Подсчёт оплат для каждой пары: клиент-месяц
monthly_counts = (
    activity_payments.groupby(['client_id', 'month'], observed=True)
    .size().rename('successful_count')
)
monthly_index = pd.MultiIndex.from_product(
    [activity_clients['client_id'], ACTIVITY_MONTHS], names=['client_id', 'month']
)
# Заполнение пропусков нулями
client_month_activity = monthly_counts.reindex(monthly_index, fill_value=0).reset_index()
# Учитываем дату регистрации
client_month_activity['registration_date'] = client_month_activity['client_id'].map(registration_by_id)
month_start = client_month_activity['month'].dt.to_timestamp()

client_month_activity['month_available'] = (
    client_month_activity['registration_date'].notna()
    & client_month_activity['registration_date'].le(month_start)
)
# successful_count_observed — найденные операции с учётом доступности месяца(регистрации).
client_month_activity['successful_count_observed'] = (
    client_month_activity['successful_count'].astype('Int64')
    .where(client_month_activity['month_available'])
)

display(client_month_activity.head(5))

## 3.4 Активность 
Будем считать период '2024-01'-'2024-12' - базовым(base), а '2025-01':'2025-06' - текущим(current). Для анализа динамики.
активность = число оплат/число месяцев

In [ ]:
# Одна строка = один клиент; каждый месяц - столбец
counts_wide = (
    client_month_activity.pivot(index='client_id', columns='month', values='successful_count')
    .reindex(columns=ACTIVITY_MONTHS, fill_value=0)
)
base_counts = counts_wide.loc[:, '2024-01':'2024-12']
current_counts = counts_wide.loc[:, '2025-01':'2025-06']
metrics = pd.DataFrame(index=counts_wide.index)
# Счётчик оплат
metrics['payments_base'] = base_counts.sum(axis=1)
metrics['payments_current'] = current_counts.sum(axis=1)
# Счётчик активности
metrics['active_months_base'] = base_counts.gt(0).sum(axis=1)
metrics['active_months_current'] = current_counts.gt(0).sum(axis=1)
metrics['activity_last3'] = counts_wide.loc[:, '2025-04':'2025-06'].sum(axis=1).div(3)
metrics['no_payments_last3'] = counts_wide.loc[:, '2025-04':'2025-06'].sum(axis=1).eq(0)

# Объединим метрику и клиентов
client_activity = activity_clients.merge(metrics.reset_index(), on='client_id', how='left', validate='one_to_one')
# Средняя активность за месяц
comparable = client_activity['activity_comparable']
client_activity['activity_base'] = client_activity['payments_base'].div(12).where(comparable)
client_activity['activity_current'] = client_activity['payments_current'].div(6).where(comparable)
client_activity['activity_last3'] = client_activity['activity_last3'].where(comparable)
# Оценим абсолютное и относительное изменение
client_activity['activity_change'] = client_activity['activity_current'] - client_activity['activity_base']
positive_base = comparable & client_activity['activity_base'].gt(0)
client_activity['activity_change_pct'] = (
    client_activity['activity_current'].div(client_activity['activity_base'].where(positive_base))
    .sub(1).mul(100)
)

# Сравниваем целые числа без ошибок округления:
# current / 6 < base / 12 эквивалентно 2 * current < base.
base = client_activity['payments_base']
current = client_activity['payments_current']
client_activity['activity_state'] = np.select(
    [~comparable, base.eq(0) & current.eq(0), base.eq(0) & current.gt(0),
     base.gt(0) & current.eq(0), current.mul(2).lt(base), current.mul(2).eq(base)],
    ['Недостаточно сопоставимой истории', 'Нет оплат в обоих периодах',
     'Появилась активность', 'Прекратились оплаты', 'Снижение', 'Без изменения'],
    default='Рост',
)
# Флаг снижения
client_activity['activity_declined'] = ( 
    comparable & base.gt(0) & current.mul(2).lt(base)
)
display(client_activity[[
    'client_id', 'activity_history_group', 'payments_base', 'payments_current',
    'activity_base', 'activity_current', 'activity_change', 'activity_change_pct',
    'activity_state',
]].head(5))


## 3.5 Расчёт и результаты


In [ ]:
# Сверка счётчиков без изменения данных
count_check = pd.DataFrame({
    'По операциям': [
        len(activity_payments),
        int(activity_payments['txn_datetime'].lt(CURRENT_START).sum()),
        int(activity_payments['txn_datetime'].ge(CURRENT_START).sum()),
    ],
    'По клиентским метрикам': [
        int((client_activity['payments_base'] + client_activity['payments_current']).sum()),
        int(client_activity['payments_base'].sum()),
        int(client_activity['payments_current'].sum()),
    ],
}, index=['Весь период', '2024 год', 'Январь–июнь 2025'])
count_check['Совпадает'] = count_check['По операциям'].eq(count_check['По клиентским метрикам'])
display(count_check)


state_summary = client_activity['activity_state'].value_counts().to_frame('Клиентов')
state_summary['Доля всей базы, %'] = state_summary['Клиентов'].div(len(client_activity)).mul(100)
display(state_summary.round(2))

comparison_sample = client_activity.loc[client_activity['activity_comparable']].copy()
display(comparison_sample[[
    'activity_base', 'activity_current', 'activity_change', 'activity_change_pct'
]].describe().round(2))

# Распределение метрик
eligible_base = comparison_sample['payments_base'].gt(0)
n_base = int(eligible_base.sum())
n_declined = int(comparison_sample['activity_declined'].sum())
n_stopped = int(comparison_sample['activity_state'].eq('Прекратились оплаты').sum())

print (f' Клиентов с историей {len(comparison_sample):,}.\n Клиенты с успешными операциями в базовом периоде: {n_base:,}.\n Клиенты без успешных операций за 2025: {n_stopped:,}' )



In [ ]:
# Распространённость состояний среди сопоставимых клиентов с положительной базой
state_plot_sample = client_activity.loc[
    client_activity['activity_comparable']
    & client_activity['payments_base'].gt(0)
].copy()

state_order = [
    'Снижение',
    'Прекратились оплаты',
    'Без изменения',
    'Рост',
]

state_labels = {
    'Снижение': 'Снижение; оплаты продолжаются',
    'Прекратились оплаты': 'Нет оплат за январь–июнь 2025',
    'Без изменения': 'Частота не изменилась',
    'Рост': 'Частота выросла',
}

state_plot_table = (
    state_plot_sample['activity_state']
    .value_counts()
    .reindex(state_order, fill_value=0)
    .rename_axis('Состояние')
    .to_frame('Клиентов')
)

n_state_plot = len(state_plot_sample)

state_plot_table['Доля, %'] = (
    state_plot_table['Клиентов']
    .div(n_state_plot if n_state_plot else np.nan)
    .mul(100)
)

if n_state_plot:
    fig, ax = plt.subplots(figsize=(10, 4.5))

    labels = [
        state_labels[state]
        for state in state_plot_table.index
    ]

    bars = ax.barh(
        labels,
        state_plot_table['Доля, %'],
        color=[
            'steelblue',
            'crimson',
            'gray',
            'darkorange',
        ],
    )

    bar_labels = [
        f'{int(row["Клиентов"]):,} клиентов — {row["Доля, %"]:.2f}%'
        for _, row in state_plot_table.iterrows()
    ]

    ax.bar_label(
        bars,
        labels=bar_labels,
        padding=5,
        fontsize=9,
    )

    ax.set(
        title=(
            'Изменение месячной частоты успешных оплат\n'
            f'Сопоставимые клиенты с положительной базой: '
            f'n = {n_state_plot:,}'
        ),
        xlabel='Доля клиентов с положительной базой, %',
        ylabel='',
        xlim=(0, 100),
    )

    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)

    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()

else:
    print(
        'Нет сопоставимых клиентов с положительной базой '
        'для построения графика.'
    )

Вывод: Сопоставимая история доступна у 5'542 клиентов. Из них 5'240 имели успешные оплаты в базовом периоде. 
У 5 клиентов с положительной базой нет успешных оплат за январь–июнь 2025 года. нижение явно перевешивает рост 67.84% к 30.59%

## 3.6 Проверка формы (диаграмма рассеяния)

In [ ]:
comparison_ids = comparison_sample['client_id']
if len(comparison_ids):
    fixed_monthly = counts_wide.loc[comparison_ids].mean(axis=0)
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
    x_months = fixed_monthly.index.to_timestamp()
    axes[0].plot(x_months, fixed_monthly.values, marker='o')
    axes[0].axvline(CURRENT_START, color='gray', linestyle='--', label='Начало оцениваемого периода')
    axes[0].set(title=f'Фиксированная группа: n = {len(comparison_ids):,}',
                xlabel='Месяц', ylabel='Успешных оплат на клиента за месяц')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend(fontsize=8)
    axes[1].scatter(comparison_sample['activity_base'], comparison_sample['activity_current'],
                    s=12, alpha=0.3)
    upper = max(comparison_sample['activity_base'].max(), comparison_sample['activity_current'].max(), 1)
    axes[1].plot([0, upper], [0, upper], color='darkorange', linestyle='--', label='Без изменения')
    axes[1].set(title='Частота оплат: базовый и текущий периоды',
                xlabel='2024: оплат в месяц', ylabel='Январь–июнь 2025: оплат в месяц')
    axes[1].legend()
    for ax in axes:
        sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()
else:
    print('Нет клиентов с полной историей: прежде чем продолжать, пересмотрите доступность сравнения.')


Вывод: помесячная линия показывает изменение частоты в фиксированной группе. На диаграмме сравнения точки ниже диагонали соответствуют снижению, выше — росту. 
- По форме линии можно предположить сезонность, так как оценивая форму '2024-01'-'2024-05' и '2025-01'-'2025-05' видно сходство, но слишком мало данных(период) для подтверждения этой гипотезы.
- По диаграмме рассеяния видно, есть и снижение и рост, основная часть клиентов сосредоточена в области небольших частот. 

In [ ]:
if 'add_log' in globals():
    add_log(
        key='clients_activity_metrics',
        field='activity_base; activity_current; activity_change; activity_change_pct; activity_state',
        issue='Нужно определить и сопоставить месячную платёжную активность',
        scope='Все клиенты; основное сравнение для регистрации не позднее 01.01.2024',
        n_rows=len(client_activity),
        value_before='Клиентские метрики активности отсутствовали',
        action='Созданы client_activity и клиентско-месячная таблица; исходные данные сохранены',
        value_after=(f'{len(client_activity)} клиентов; {len(comparison_sample)} сопоставимы; '
                     f'{len(client_month_activity)} клиентско-месячных строк'),
        rationale=('Успешные положительные конечные оплаты по всем каналам и валютам. '
                   'Для полной истории количество делится на 12 и 6 месяцев; '
                   'месяцы без оплат учитываются нулями. При нулевой базе процент не определяется. '
                   'Недостаточная история не является основанием удаления клиента.'),
    )
else:
    print('add_log не определена: выполните инициализацию журнала и повторите эту ячейку.')


## Итог
Определена основная метрика активности: среднее количество оплат за месяц.
Снижение активности преобладает над ростом

Преимущества метрики удобно сравнивать разные периоды времени и легко разделяется по признакам(категориям), например, категория трат, способ.

Недостатки метрики не показывает изменения сумм операций, не учитывает операции в других банках. Усреднение может не отражать прекращение оплат в определённый период месяца(заканчивается зарплата к концу месяца - денег нет)